# Tratamento de dados — Amazon

## 1 — Do diagnóstico estrutural ao modelo de dados

### 1.1 — O que o Notebook 01 demonstrou

A fonte reúne 1.465 ocorrências e 1.351 IDs de produto. O caminho de categoria é estável por ID. As listas de códigos se alinham por posição, mas o texto possui pontuação interna. Essas evidências determinam os grãos e os testes abaixo.

### 1.2 — Quais entidades precisamos construir?

Ocorrências retêm a origem literal. Produtos trazem a referência comercial por ID. Categorias representam caminhos. Usuários e Reviews individualizam códigos. Vínculos registram cada posição do review em sua ocorrência.

### 1.3 — Modelo entidade-relacionamento

Categorias → Produtos → Ocorrências → Vínculos. Usuários → Reviews → Vínculos. As referências comerciais e textuais voltam à ocorrência de origem. O ER físico é gerado na seção 7 a partir dos tipos finais.

### 1.4 — Regras de transformação e integridade

Conversões e cenários analíticos ficam separados da fonte. A seleção comercial reutiliza uma única linha real. Os candidatos textuais ficam em auditoria até que a evidência sustente a atribuição. A exportação é para staging ou proposta.

## 2 — Preservando e preparando a origem

### 2.1 — Leitura dos dados originais

A leitura preserva os 16 campos literais e interpreta como nulo somente o campo vazio.

In [1]:
from pathlib import Path
import os
import hashlib

PUBLICAR_CSVS = False  # Opt-in manual; manter False para validação.
import pandas as pd
from IPython.display import display

raiz = next(c for p in [Path.cwd(), *Path.cwd().parents]
            for c in [p, p / "amazon_sales_analise"] if (c / "amazon.csv").exists())
# Somente o campo vazio é ausência: textos como "NA" não são reinterpretados.
fonte = pd.read_csv(raiz / "amazon.csv", dtype="string", keep_default_na=False, na_values=[""])
observacoes = fonte.copy()
observacoes.insert(0, "source_index", pd.Series(range(len(fonte)), dtype="Int64"))
frequencias_produtos = fonte["product_id"].value_counts()
display(pd.Series({"registros": len(fonte), "campos_originais": fonte.shape[1],
    "produtos": len(frequencias_produtos), "IDs_repetidos": frequencias_produtos.gt(1).sum(),
    "linhas_desses_IDs": frequencias_produtos[frequencias_produtos.gt(1)].sum(),
    "duplicatas_integrais": fonte.duplicated().sum()}).to_frame("quantidade"))

pasta_oficial = (raiz / "dados_tratados").resolve()
hashes_oficiais_antes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                        for p in pasta_oficial.glob("*.csv")}

def mostrar_esquema(tabela):
    """Esquema de exibição: tipos, campos e ausências, sem modificar a tabela."""
    esquema = pd.DataFrame({"campo": tabela.columns, "tipo": tabela.dtypes.astype(str).values,
                            "nulos": tabela.isna().sum().values})
    compacto = esquema.groupby("tipo", sort=False).agg(
        campos=("campo", ", ".join), quantidade=("campo", "size"))
    display(compacto.style.set_properties(**{"max-width": "650px", "white-space": "normal", "overflow-wrap": "anywhere"}))
    display(esquema.loc[esquema["nulos"].gt(0), ["campo", "nulos"]].reset_index(drop=True))

,quantidade
registros,1465
campos_originais,16
produtos,1351
IDs_repetidos,92
linhas_desses_IDs,206
duplicatas_integrais,0


### 2.2 — Identificação das ocorrências

O código OC deriva da posição original com base um. A posição técnica preservada no campo source_index usa base zero.

In [2]:
ocorrencias_base = observacoes.copy()
ocorrencias_base.insert(0, 'ocorrencia_id', 'OC-' + (observacoes.source_index+1).astype('string').str.zfill(6))
display(ocorrencias_base[['ocorrencia_id','source_index','product_id']].head())

,ocorrencia_id,source_index,product_id
0,OC-000001,0,B07JW9H4J1
1,OC-000002,1,B098NS6PVG
2,OC-000003,2,B096MSW6CT
3,OC-000004,3,B08HDJ86NZ
4,OC-000005,4,B08CF3B7N1


### 2.3 — Tratamento dos valores comerciais

Preços são convertidos para INR, descontos para pontos percentuais na escala 0–100, e contagens para inteiros. Os exemplos incluem os formatos de exceção.

In [3]:
campos_numericos_brutos = ["actual_price", "discounted_price", "discount_percentage", "rating", "rating_count"]
indices_exemplo = list(dict.fromkeys([0,
    fonte.index[fonte["discounted_price"].str.contains(".", regex=False)][0],
    fonte.index[fonte["rating_count"].str.count(",").ge(2)][0],
    *fonte.index[fonte["rating"].eq("|") | fonte["rating_count"].isna()]]))
display(observacoes.loc[indices_exemplo, ["source_index", "product_id"] + campos_numericos_brutos])

,source_index,product_id,actual_price,discounted_price,discount_percentage,rating,rating_count
0,0,B07JW9H4J1,"₹1,099",₹399,64%,4.2,"24,269"
6,6,B08WRWPM22,₹499,₹176.63,65%,4.1,"15,188"
8,8,B008IFXQFU,₹999,₹499,50%,4.2,"1,79,691"
282,282,B0B94JPY2N,₹999,₹199,80%,3.0,<NA>
324,324,B0BQRJ3C47,₹999,₹249,75%,5.0,<NA>
1279,1279,B08L12N5H1,"₹2,499","₹2,099",16%,|,992


In [4]:
for campo in ["actual_price", "discounted_price"]:
    limpo = observacoes[campo].str.replace("₹", "", regex=False).str.replace(",", "", regex=False)
    observacoes[campo + "_num"] = pd.to_numeric(limpo, errors="coerce").astype("Float64")
observacoes["discount_percentage_num"] = pd.to_numeric(
    observacoes["discount_percentage"].str.replace("%", "", regex=False), errors="coerce").astype("Float64")
notas = pd.to_numeric(observacoes["rating"], errors="coerce").astype("Float64")
observacoes["rating_num"] = notas.where(notas.between(1, 5))
observacoes["rating_invalido"] = observacoes["rating"].notna() & observacoes["rating_num"].isna()
contagens = observacoes["rating_count"].str.replace(",", "", regex=False)
observacoes["rating_count_num"] = pd.to_numeric(
    contagens.where(contagens.str.fullmatch(r"\d+", na=False)), errors="coerce"
).astype("Int64")
observacoes["rating_count_ausente"] = observacoes["rating_count"].isna()
observacoes["rating_count_situacao"] = "Informado"
observacoes.loc[observacoes["rating_count_num"].isna(), "rating_count_situacao"] = "Formato inválido"
observacoes.loc[observacoes["rating_count_ausente"], "rating_count_situacao"] = "Não informado"
display(pd.concat({campo: observacoes.loc[indices_exemplo, [campo, campo + "_num"]]
                   .rename(columns={campo: "original", campo + "_num": "numerico"}).astype("string")
                   for campo in campos_numericos_brutos}, names=["campo", "source_index"]))

original numerico
campo               source_index                   
actual_price        0               ₹1,099   1099.0
                    6                 ₹499    499.0
                    8                 ₹999    999.0
                    282               ₹999    999.0
                    324               ₹999    999.0
                    1279            ₹2,499   2499.0
discounted_price    0                 ₹399    399.0
                    6              ₹176.63   176.63
                    8                 ₹499    499.0
                    282               ₹199    199.0
                    324               ₹249    249.0
                    1279            ₹2,099   2099.0
discount_percentage 0                  64%     64.0
                    6                  65%     65.0
                    8                  50%     50.0
                    282                80%     80.0
                    324                75%     75.0
                    1279               16%     16.0
rating              0                  4.2      4.2
                    6                  4.1      4.1
                    8                  4.2      4.2
                    282                3.0      3.0
                    324                5.0      5.0
                    1279                 |     <NA>
rating_count        0               24,269    24269
                    6               15,188    15188
                    8             1,79,691   179691
                    282               <NA>     <NA>
                    324               <NA>     <NA>
                    1279               992      992

In [5]:
observacoes["desconto_calculado_num"] = (
    1 - observacoes["discounted_price_num"] / observacoes["actual_price_num"].where(observacoes["actual_price_num"].gt(0))
) * 100
observacoes["diferenca_desconto_pp"] = (observacoes["discount_percentage_num"] - observacoes["desconto_calculado_num"]).abs()
observacoes["desconto_nao_avaliado"] = observacoes["diferenca_desconto_pp"].isna()
# O arredondamento em 10 casas evita classificar ruído numérico como desvio do limite.
observacoes["desconto_inconsistente"] = observacoes["diferenca_desconto_pp"].round(10).gt(0.5).fillna(False)
display(observacoes.nlargest(5, "diferenca_desconto_pp")[["source_index", "discount_percentage_num",
    "desconto_calculado_num", "diferenca_desconto_pp", "desconto_inconsistente"]].round(4))

,source_index,discount_percentage_num,desconto_calculado_num,diferenca_desconto_pp,desconto_inconsistente
817,817,59.0,58.5,0.5,False
1456,1456,59.0,58.5,0.5,False
490,490,13.0,12.5008,0.4992,False
522,522,13.0,12.5008,0.4992,False
1075,1075,14.0,14.4984,0.4984,False


### 2.4 — Conferência das transformações iniciais

A tolerância de desconto é de 0,5 ponto percentual. A nota inválida e as contagens ausentes permanecem nulas no observado.

In [6]:
pd.testing.assert_frame_equal(ocorrencias_base[fonte.columns],fonte)
assert observacoes.actual_price_num.gt(0).all()
assert observacoes.discounted_price_num.between(0,observacoes.actual_price_num).all()
assert observacoes.discount_percentage_num.between(0,100).all()
assert not observacoes[['desconto_inconsistente','desconto_nao_avaliado']].any().any()
display(observacoes[['rating_invalido','desconto_inconsistente']].sum().to_frame('ocorrências'))

,ocorrências
rating_invalido,1
desconto_inconsistente,0


## 3 — Construindo a entidade Produtos

### 3.1 — Identidade e repetição dos produtos

As divergências dentro do mesmo ID explicam por que precisamos escolher uma observação inteira e conservar as alternativas.

In [7]:
campos_comerciais = ["actual_price_num", "discounted_price_num", "discount_percentage_num", "rating_num", "rating_count_num"]
campos_comparados = ["product_name", "category"] + campos_comerciais + ["about_product", "img_link"]
conflitos_por_produto = observacoes.groupby("product_id")[campos_comparados].nunique(dropna=False).gt(1)


display(conflitos_por_produto.sum().sort_values(ascending=False).to_frame("produtos_com_divergencia"))

amplitudes = observacoes.groupby("product_id")[campos_comerciais].max() - observacoes.groupby("product_id")[campos_comerciais].min()
ids_conflitos_numericos = amplitudes[["actual_price_num", "discounted_price_num", "rating_count_num"]].idxmax().unique()

display(observacoes.loc[observacoes["product_id"].isin(ids_conflitos_numericos), [
    "source_index", "product_id", "actual_price_num", "discounted_price_num", "discount_percentage_num", "rating_count_num",
]].sort_values(["product_id", "source_index"]))

,produtos_com_divergencia
img_link,77
rating_count_num,31
discount_percentage_num,4
about_product,4
discounted_price_num,3
actual_price_num,1
product_name,0
category,0
rating_num,0


,source_index,product_id,actual_price_num,discounted_price_num,discount_percentage_num,rating_count_num
2,2,B096MSW6CT,1899.0,199.0,90.0,7928
379,379,B096MSW6CT,999.0,199.0,80.0,7928
623,623,B096MSW6CT,999.0,199.0,80.0,7928
335,335,B0B3RRWSF6,9999.0,1998.0,80.0,27696
586,586,B0B3RRWSF6,9999.0,1998.0,80.0,27709
336,336,B0B5B6PQCT,7990.0,1999.0,75.0,17831
587,587,B0B5B6PQCT,7990.0,1799.0,77.0,17833


### 3.2 — Deduplicação e equivalências candidatas

A igualdade integral e a igualdade em 13 campos são verificações distintas. Uma fusão dos grupos candidatos reduziria o cadastro pelo número de IDs excedentes exibido. A fonte não estabelece equivalência comercial.

In [8]:
campos_similaridade = fonte.columns.drop(["product_id", "product_link", "img_link"]).tolist()
grupos_iguais = observacoes.groupby(campos_similaridade, dropna=False, sort=False)
evidencias_similaridade = observacoes[["product_id", "source_index"]].copy()
evidencias_similaridade["origem_grupo"] = grupos_iguais["source_index"].transform("min")
evidencias_similaridade["quantidade_ids_grupo"] = grupos_iguais["product_id"].transform("nunique")
similaridade_produtos = evidencias_similaridade.loc[evidencias_similaridade["quantidade_ids_grupo"].gt(1)].copy()
similaridade_produtos["grupo_equivalencia_registros_id"] = "E13-" + similaridade_produtos["origem_grupo"].astype(str).str.zfill(4)
similaridade_produtos = similaridade_produtos.sort_values("source_index").drop_duplicates(["product_id", "grupo_equivalencia_registros_id"])
similaridade_produtos = similaridade_produtos.rename(columns={"source_index": "source_index_evidencia"}).drop(columns="origem_grupo")
similaridade_produtos["criterio_similaridade"] = "equivalencia_exata_13_campos"
similaridade_produtos = similaridade_produtos.reset_index(drop=True)
equivalencias_registros = similaridade_produtos
assert equivalencias_registros["product_id"].is_unique
display(equivalencias_registros)
display(equivalencias_registros.groupby("grupo_equivalencia_registros_id")["product_id"].nunique().to_frame("produtos"))

,product_id,source_index_evidencia,quantidade_ids_grupo,grupo_equivalencia_registros_id,criterio_similaridade
0,B09L835C3V,243,2,E13-0243,equivalencia_exata_13_campos
1,B09HK9JH4F,296,2,E13-0243,equivalencia_exata_13_campos
2,B0BF57RN3K,334,2,E13-0334,equivalencia_exata_13_campos
3,B0BF54972T,364,2,E13-0364,equivalencia_exata_13_campos
4,B09TWH8YHM,366,2,E13-0366,equivalencia_exata_13_campos
5,B0BF563HB4,368,2,E13-0364,equivalencia_exata_13_campos
6,B0BF54LXW6,380,2,E13-0334,equivalencia_exata_13_campos
7,B0B4F3QNDM,399,3,E13-0399,equivalencia_exata_13_campos
8,B0B14MR9L1,411,2,E13-0366,equivalencia_exata_13_campos
9,B0B4F5L738,425,3,E13-0399,equivalencia_exata_13_campos


,produtos
grupo_equivalencia_registros_id,
E13-0243,2
E13-0334,2
E13-0364,2
E13-0366,2
E13-0399,3
E13-0501,2


In [9]:
print('Redução hipotética de IDs:',len(equivalencias_registros)-equivalencias_registros.grupo_equivalencia_registros_id.nunique())

Redução hipotética de IDs: 7


### 3.3 — Escolha da ocorrência comercial de referência

A prioridade é completude observada, frequência da combinação e menor posição original. A regra não pressupõe que o menor índice seja mais recente.

In [10]:
# Preparação de campos opcionais para o cadastro. O cenário é calculado após a seleção.
for campo in ['rating','rating_count']:
    observacoes[campo+'_analitico_num'] = observacoes[campo+'_num']
    observacoes[campo+'_estimado'] = False
    observacoes['metodo_estimativa_'+campo] = pd.Series(pd.NA,index=observacoes.index,dtype='string')

In [11]:
candidatas = observacoes.copy()
candidatas["completude_observada"] = candidatas[campos_comerciais].notna().sum(axis=1)
candidatas["frequencia_combinacao"] = candidatas.groupby(
    ["product_id"] + campos_comerciais, dropna=False
)["source_index"].transform("size")
referencias_produtos = candidatas.sort_values(
    ["completude_observada", "frequencia_combinacao", "source_index"], ascending=[False, False, True]
).drop_duplicates("product_id")
melhores_completudes = candidatas.loc[candidatas["completude_observada"].eq(
    candidatas.groupby("product_id")["completude_observada"].transform("max"))]
candidatas_empatadas = melhores_completudes.loc[melhores_completudes["frequencia_combinacao"].eq(
    melhores_completudes.groupby("product_id")["frequencia_combinacao"].transform("max"))]
combinacoes_empatadas = candidatas_empatadas.drop_duplicates(["product_id"] + campos_comerciais).groupby("product_id").size()
assert candidatas.groupby("product_id")["completude_observada"].nunique().eq(1).all()
colunas_produto = [
    "product_id", "product_name", "category", "about_product", "product_link", "img_link", "source_index",
] + campos_comerciais + [
    "rating_analitico_num", "rating_estimado", "metodo_estimativa_rating",
    "rating_count_analitico_num", "rating_count_estimado", "metodo_estimativa_rating_count", "rating_count_situacao",
]
renomes_produtos = {
    "source_index": "source_index_referencia", "category": "category_original",
    "about_product": "about_product_referencia", "product_link": "product_link_referencia", "img_link": "img_link_referencia",
}
produtos = referencias_produtos[colunas_produto].rename(columns=renomes_produtos).copy()
produtos = produtos.join(frequencias_produtos.rename("quantidade_observacoes"), on="product_id")
produtos = produtos.sort_values("product_id").reset_index(drop=True)
conflitos_uteis = conflitos_por_produto[[
    "actual_price_num", "discounted_price_num", "discount_percentage_num", "rating_count_num", "about_product", "img_link",
]].rename(columns={
    "actual_price_num": "conflito_preco_original", "discounted_price_num": "conflito_preco_desconto",
    "discount_percentage_num": "conflito_percentual", "rating_count_num": "conflito_contagem",
    "about_product": "conflito_descricao", "img_link": "conflito_imagem",
})
produtos = produtos.join(conflitos_uteis, on="product_id")

origens_referenciadas = observacoes.set_index("source_index").loc[produtos["source_index_referencia"]].reset_index()
valores_representativos = produtos.rename(columns={destino: origem for origem, destino in renomes_produtos.items()})
assert valores_representativos[colunas_produto].equals(origens_referenciadas[colunas_produto]), "Valores de origens diferentes foram combinados."
assert produtos["product_id"].is_unique and set(produtos["product_id"]) == set(observacoes["product_id"]), "A cobertura dos IDs mudou."
assert produtos["quantidade_observacoes"].sum() == len(observacoes), "As contagens de observações não se conciliam."
quantidade_combinacoes = candidatas.drop_duplicates(["product_id"] + campos_comerciais).groupby("product_id").size()
produtos["criterio_selecao_referencia"] = "Desempate pela ordem da fonte"
produtos.loc[
    produtos["product_id"].map(quantidade_combinacoes).gt(1)
    & produtos["product_id"].map(combinacoes_empatadas).eq(1), "criterio_selecao_referencia"
] = "Combinação comercial mais frequente"
produtos.loc[
    produtos["product_id"].map(quantidade_combinacoes).eq(1), "criterio_selecao_referencia"
] = "Mesma combinação comercial"
produtos.loc[produtos["quantidade_observacoes"].eq(1), "criterio_selecao_referencia"] = "Ocorrência única"

distribuicao_criterios = produtos["criterio_selecao_referencia"].value_counts().to_frame("produtos")
display(distribuicao_criterios)

,produtos
criterio_selecao_referencia,
Ocorrência única,1259
Mesma combinação comercial,59
Desempate pela ordem da fonte,24
Combinação comercial mais frequente,9


In [12]:
ocorrencias_catalogo = observacoes.assign(ocorrencia_id=ocorrencias_base.ocorrencia_id)
display(candidatas.loc[candidatas.product_id.eq('B09MT84WV5'),
    ['source_index','discounted_price_num','rating_count_num','completude_observada','frequencia_combinacao']]
    .assign(referencia=lambda x:x.source_index.isin(produtos.source_index_referencia)))

,source_index,discounted_price_num,rating_count_num,completude_observada,frequencia_combinacao,referencia
397,397,1149.0,140036,5,1,True
641,641,1059.0,140035,5,1,False


### 3.4 — Construção do cadastro comercial

A referência OC pode ser usada para recuperar todos os valores brutos da observação selecionada.

In [13]:
produtos['ocorrencia_id_referencia'] = produtos.source_index_referencia.map(ocorrencias_base.set_index('source_index').ocorrencia_id)
display(produtos[['product_id','ocorrencia_id_referencia','actual_price_num','discounted_price_num','rating_num']].head())

,product_id,ocorrencia_id_referencia,actual_price_num,discounted_price_num,rating_num
0,B002PD61Y4,OC-000047,1208.0,507.0,4.1
1,B002SZEOLG,OC-000144,1339.0,749.0,4.2
2,B003B00484,OC-000722,499.0,399.0,4.3
3,B003L62T7W,OC-000725,375.0,279.0,4.3
4,B004IO5BMQ,OC-000735,995.0,699.0,4.5


### 3.5 — Conflitos, proveniência e validação

Os indicadores de conflito sinalizam a existência de alternativas, que continuam presentes em Ocorrências.

In [14]:
assert produtos.product_id.equals(produtos.ocorrencia_id_referencia.map(ocorrencias_base.set_index('ocorrencia_id').product_id))
display(conflitos_uteis.sum().to_frame('produtos'))

,produtos
conflito_preco_original,1
conflito_preco_desconto,3
conflito_percentual,4
conflito_contagem,31
conflito_descricao,4
conflito_imagem,77


## 4 — Construindo a entidade Categorias

### 4.1 — Separação da hierarquia original

Cada caminho literal define uma categoria. A barra vertical separa níveis e a vírgula pode pertencer ao nome.

In [15]:
categorias = fonte[['category']].drop_duplicates().sort_values('category').reset_index(drop=True).rename(columns={'category':'category_original'})
partes = categorias.category_original.str.split('|',regex=False)
display(partes.str.len().value_counts().sort_index().rename_axis('níveis').to_frame('caminhos'))

,caminhos
níveis,
2,4
3,33
4,106
5,49
6,12
7,7


### 4.2 — Criação dos identificadores de categoria

SHA-256 do caminho integral em UTF-8 fornece uma chave independente da ordem das linhas. A unicidade é conferida na dimensão.

In [16]:
categorias.insert(0,'categoria_id',categorias.category_original.map(lambda s:'CAT-'+hashlib.sha256(s.encode('utf-8')).hexdigest()).astype('string'))
assert categorias.categoria_id.is_unique

### 4.3 — Macro original, níveis e categoria específica

A primeira parte é a macro original. A última parte existente é a categoria específica. As posições além da profundidade ficam nulas.

In [17]:
niveis = [f'categoria_nivel_{i}' for i in range(1,8)]
for i,campo in enumerate(niveis):
    categorias[campo] = partes.str[i].astype('string')
categorias['categoria_macro_original'] = partes.str[0].astype('string')
categorias['categoria_especifica'] = partes.str[-1].astype('string')
categorias['profundidade_categoria'] = partes.str.len().astype('Int64')

### 4.4 — Relacionamento com Produtos

A associação usa o caminho original completo e produz a FK categoria_id no cadastro.

In [18]:
produtos['categoria_id'] = produtos.category_original.map(categorias.set_index('category_original').categoria_id)
assert produtos.categoria_id.notna().all()

### 4.5 — Preparação da categoria macro por ML

Método, versão e saída ficam nulos até a definição e avaliação de uma taxonomia comercial.

In [19]:
for campo in ['categoria_macro_ml','metodo_macro_ml','versao_macro_ml']:
    categorias[campo] = pd.Series(pd.NA,index=categorias.index,dtype='string')
categorias['status_macro_ml'] = 'PENDENTE'

### 4.6 — Validação dos filtros e da hierarquia

Rótulos terminais homônimos permanecem em caminhos distintos. Recompomos os caminhos e testamos as quantidades de produtos por macro.

In [20]:
assert categorias[niveis].apply(lambda row:'|'.join(row.dropna()),axis=1).tolist()==categorias.category_original.tolist()
display(categorias.groupby('categoria_macro_original').size().to_frame('caminhos'))
display(categorias.loc[categorias.categoria_especifica.eq('Adapters'),['category_original','profundidade_categoria']])

,caminhos
categoria_macro_original,
Car&Motorbike,1
Computers&Accessories,57
Electronics,62
Health&PersonalCare,1
Home&Kitchen,72
HomeImprovement,2
MusicalInstruments,1
OfficeProducts,14
Toys&Games,1


,category_original,profundidade_categoria
73,"Electronics|Headphones,Earbuds&Accessories|Ada...",3
79,Electronics|HomeAudio|Accessories|Adapters,4


## 5 — Construindo a entidade Usuários

### 5.1 — Individualização dos identificadores

As listas de códigos preservam a ordem original. O exemplo contrasta ocorrências unitárias, múltiplas e nomes ambíguos.

In [21]:
quantidades_usuarios = ocorrencias_catalogo["user_id"].str.split(",", regex=False).str.len()
quantidades_nomes = ocorrencias_catalogo["user_name"].str.split(",", regex=False).str.len()
nomes_compativeis = quantidades_usuarios.eq(quantidades_nomes)
indices_usuarios = [quantidades_usuarios[quantidades_usuarios.eq(1)].index[0],
    quantidades_usuarios[quantidades_usuarios.gt(1)].index[0], nomes_compativeis[~nomes_compativeis].index[0]]
display(ocorrencias_catalogo.loc[indices_usuarios, ["source_index", "user_id", "user_name"]].T.style.set_properties(**{"max-width": "320px", "white-space": "normal", "overflow-wrap": "anywhere"}))
display(pd.DataFrame({"IDs": quantidades_usuarios, "nomes_aparentes": quantidades_nomes}).loc[indices_usuarios])

,155,0,5
source_index,155,0,5
user_id,AE242TR3GQ6TYC6W4SJ5UYYKBTYQ,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBBSNLYT3ONILA,AHCTC6ULH4XB6YHDY6PCH2R772LQ,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,AENGU523SXMOS7JPDTW52PNNVWGQ,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ","AEQ2YMXSZWEOHK2EHTNLOS56YTZQ,AGRVINWECNY7323CWFXZYYIZOFTQ,AHBAT6VLOXWGYDL57KHCNCLPXAKA,AF7NDY2H6JVYTSQOZP76GCATQ34Q,AFV7ZA733ZLME4KNLZPMPCBUNPPA,AHFAAPSY2MJ5HYOU2VQDJ7AQY4NQ,AH2WGV2PEBUTICRPBEEVKF24G5LA,AEP4MK3EKOBDKTGPJTRN5RBDIODA"
user_name,anurag jain,"Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jaspreet singh,Khaja moin,Anand,S.ARUMUGAM","Jayesh,Rajesh k.,Soopy,amazon customer,Aman,Shankar,dinesh,Chitra,Ajaybabu.O.M"


,IDs,nomes_aparentes
155,1,1
0,8,8
5,8,9


### 5.2 — Recuperação e consistência dos nomes

A regra anterior exige quantidade compatível e um único nome aparente por ID no subconjunto elegível. Isso não estabelece identidade civil.

In [22]:
tokens_usuarios = ocorrencias_catalogo[["ocorrencia_id", "user_id"]].copy()
tokens_usuarios["user_id"] = tokens_usuarios["user_id"].str.split(",", regex=False)
tokens_usuarios = tokens_usuarios.explode("user_id", ignore_index=True)
tokens_usuarios["posicao_na_linha"] = tokens_usuarios.groupby("ocorrencia_id", sort=False).cumcount() + 1
candidatos_nome = ocorrencias_catalogo.loc[nomes_compativeis, ["ocorrencia_id", "user_id", "user_name"]].copy()
for campo in ["user_id", "user_name"]:
    candidatos_nome[campo] = candidatos_nome[campo].str.split(",", regex=False)
candidatos_nome = candidatos_nome.explode(["user_id", "user_name"], ignore_index=True)
candidatos_nome["user_name"] = candidatos_nome["user_name"].str.strip().replace("", pd.NA)
perfil_nomes = candidatos_nome.groupby("user_id").agg(
    nomes_distintos=("user_name", "nunique"), nome_candidato=("user_name", "first"))
nomes_consistentes = perfil_nomes.loc[perfil_nomes["nomes_distintos"].eq(1), "nome_candidato"]
display(candidatos_nome.head(8))
usuarios = tokens_usuarios[["user_id"]].drop_duplicates().sort_values("user_id").reset_index(drop=True).astype("string")
usuarios["user_name_recuperado"] = usuarios["user_id"].map(nomes_consistentes).astype("string")
ids_ambiguos = tokens_usuarios.loc[tokens_usuarios["ocorrencia_id"].isin(
    ocorrencias_catalogo.loc[~nomes_compativeis, "ocorrencia_id"]), "user_id"]
usuarios["aparece_em_nome_ambiguo"] = usuarios["user_id"].isin(ids_ambiguos)
usuarios["situacao_nome"] = "Sem nome recuperável"
usuarios.loc[usuarios["user_name_recuperado"].notna(), "situacao_nome"] = "Associação posicional consistente"
usuarios.loc[usuarios["user_id"].isin(perfil_nomes.loc[perfil_nomes["nomes_distintos"].gt(1)].index),
             "situacao_nome"] = "Nomes divergentes nas ocorrências compatíveis"
display(usuarios.groupby(["situacao_nome", "aparece_em_nome_ambiguo"]).size().to_frame("usuarios"))
display(usuarios.loc[usuarios["aparece_em_nome_ambiguo"]].sort_values("user_name_recuperado", na_position="first").head(6))
display(usuarios.loc[usuarios["aparece_em_nome_ambiguo"] & usuarios["user_name_recuperado"].notna()])
ids_por_nome = candidatos_nome.groupby("user_name")["user_id"].nunique()
display(pd.Series({"linhas_ambiguas": (~nomes_compativeis).sum(),
    "nomes_recuperados": usuarios["user_name_recuperado"].notna().sum(),
    "sem_nome": usuarios["user_name_recuperado"].isna().sum(),
    "nomes_em_varios_IDs": ids_por_nome.gt(1).sum(), "IDs_Amazon_Customer": ids_por_nome.loc["Amazon Customer"]}).to_frame("quantidade"))

,ocorrencia_id,user_id,user_name
0,OC-000001,AG3D6O4STAQKAY2UVGEUV46KN35Q,Manav
1,OC-000001,AHMY5CWJMMK5BJRBBSNLYT3ONILA,Adarsh gupta
2,OC-000001,AHCTC6ULH4XB6YHDY6PCH2R772LQ,Sundeep
3,OC-000001,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,S.Sayeed Ahmed
4,OC-000001,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,jaspreet singh
5,OC-000001,AENGU523SXMOS7JPDTW52PNNVWGQ,Khaja moin
6,OC-000001,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,Anand
7,OC-000001,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ,S.ARUMUGAM


usuarios
situacao_nome                     aparece_em_nome_ambiguo          
Associação posicional consistente False                        8994
                                  True                            3
Sem nome recuperável              True                           53

,user_id,user_name_recuperado,aparece_em_nome_ambiguo,situacao_nome
129,AE3XNUB7K2MKUGB4H4KY2JY7WVQQ,<NA>,True,Sem nome recuperável
360,AE7BNHD6PZQQD7K3OKFEPFHTISSA,<NA>,True,Sem nome recuperável
619,AECUDCUUNINQYLQOYTKNXGMYWIDQ,<NA>,True,Sem nome recuperável
1011,AEIFMHDK4ETHLYWSV6TUFNSJU4MQ,<NA>,True,Sem nome recuperável
1075,AEJG6XAZFMVQ7NRKCXG3ZCC3DIVA,<NA>,True,Sem nome recuperável
1175,AEKRUOFGND5373O77W4ZRW5H4ROQ,<NA>,True,Sem nome recuperável


,user_id,user_name_recuperado,aparece_em_nome_ambiguo,situacao_nome
2578,AF6HB6GYUYNZ4G4FDTQIGQK76WSQ,A.M.,True,Associação posicional consistente
4210,AFV7ZA733ZLME4KNLZPMPCBUNPPA,Shankar,True,Associação posicional consistente
8525,AHS5BOPH3WRQV2BD4IWZRGDYQVVQ,hemant,True,Associação posicional consistente


,quantidade
linhas_ambiguas,12
nomes_recuperados,8997
sem_nome,53
nomes_em_varios_IDs,422
IDs_Amazon_Customer,512


### 5.3 — Consolidação do cadastro de usuários

Cada código recebe uma linha. A quantidade de reviews distintos é incorporada após o pareamento.

### 5.4 — Validação dos usuários

A cobertura inclui os códigos que aparecem apenas em listas de nomes ambíguas.

In [23]:
assert usuarios.user_id.is_unique and set(usuarios.user_id)==set(tokens_usuarios.user_id)

## 6 — Reconstruindo a entidade Reviews

### 6.1 — Estrutura original dos identificadores e textos

O primeiro exemplo reúne oito posições. A auditoria mantém títulos e conteúdos integrais e a prévia mostra a estrutura sem abrir blocos longos.

In [24]:
comprimentos = ocorrencias_catalogo["review_content"].str.len()
indices_texto = [0, comprimentos.idxmax()]
previas_textuais = ocorrencias_catalogo.loc[indices_texto, ["ocorrencia_id", "product_id"]].copy()
previas_textuais["titulo_previa"] = ocorrencias_catalogo.loc[indices_texto, "review_title"].str.slice(0, 160)
previas_textuais["conteudo_previa"] = ocorrencias_catalogo.loc[indices_texto, "review_content"].str.slice(0, 250)
previas_textuais["caracteres_integrais"] = comprimentos.loc[indices_texto]
display(previas_textuais.T.style.set_properties(**{"max-width": "430px", "white-space": "normal", "overflow-wrap": "anywhere"}))
quantidades_reviews = ocorrencias_catalogo["review_id"].str.split(",", regex=False).str.len()
display(pd.Series({campo: ocorrencias_catalogo[campo].str.split(",", regex=False).str.len().eq(quantidades_reviews).sum()
    for campo in ["review_title", "review_content"]}).to_frame("linhas_com_quantidades_coincidentes"))

,0,1056
ocorrencia_id,OC-000001,OC-001057
product_id,B07JW9H4J1,B008QTK47Q
titulo_previa,"Satisfied,Charging is really fast,Value for money,Product review,Good quality,Good product,Good Product,As of now seems good","How to choose an iron ? This one-A decent combo of features & price.Cable quality not good though.,Quality Iron for power users with effective spray & steam fun"
conteudo_previa,"Looks durable Charging is fine tooNo complains,Charging is really fast, good product.,Till now satisfied with the quality.,This is a good product . The charging speed is slower than the original iPhone cable,Good quality, would recommend,https://m.me","Update as on 28.10.2018:*********************************The power cord has developed twists beyond repair and the seperate cores are now visible,the insulation pvc of the cores doesn't look very durable.I don't know why such standard manufacturers c"
caracteres_integrais,483,18547


,linhas_com_quantidades_coincidentes
review_title,989
review_content,197


### 6.2 — Individualização dos IDs de review

A posição recupera os pares de códigos. A consistência review–usuário é verificada entre ocorrências.

In [25]:
tokens_reviews = ocorrencias_catalogo[["ocorrencia_id", "product_id", "review_id"]].copy()
tokens_reviews["review_id"] = tokens_reviews["review_id"].str.split(",", regex=False)
tokens_reviews = tokens_reviews.explode("review_id", ignore_index=True)
avaliacoes_identificadas = tokens_reviews.groupby("review_id").agg(
    quantidade_ocorrencias=("ocorrencia_id", "size"),
    quantidade_produtos_associados=("product_id", "nunique")).reset_index()
review_exemplo = avaliacoes_identificadas.sort_values(
    ["quantidade_produtos_associados", "quantidade_ocorrencias", "review_id"], ascending=[False, False, True]).iloc[0]["review_id"]
display(avaliacoes_identificadas.head(6))
display(tokens_reviews.loc[tokens_reviews["review_id"].eq(review_exemplo)])
display(pd.Series({"aparicoes": len(tokens_reviews), "reviews": len(avaliacoes_identificadas),
    "reviews_em_varios_produtos": avaliacoes_identificadas["quantidade_produtos_associados"].gt(1).sum(),
    "pares_produto_review": len(tokens_reviews.drop_duplicates(["product_id", "review_id"]))}).to_frame("quantidade"))

,review_id,quantidade_ocorrencias,quantidade_produtos_associados
0,R100Y29EI0KGW9,1,1
1,R101TILZBOMQ6F,1,1
2,R101VJD80D1Z15,1,1
3,R10365HEDURWI9,3,2
4,R103G2OV6OFA3Q,1,1
5,R1042SYVJXWW5H,1,1


,ocorrencia_id,product_id,review_id
186,OC-000024,B09NHVCHS9,R1O4Z15FD40PV5
354,OC-000045,B09NJN8L25,R1O4Z15FD40PV5
650,OC-000082,B09NKZXMWJ,R1O4Z15FD40PV5
1390,OC-000177,B0B3N8VG24,R1O4Z15FD40PV5
1406,OC-000179,B0B3MQXNFB,R1O4Z15FD40PV5
1550,OC-000197,B08P9RYPLR,R1O4Z15FD40PV5
2004,OC-000254,B08N1WL9XW,R1O4Z15FD40PV5
3739,OC-000479,B09NHVCHS9,R1O4Z15FD40PV5
3803,OC-000487,B09NL4DJ2Z,R1O4Z15FD40PV5
5695,OC-000728,B09NHVCHS9,R1O4Z15FD40PV5


,quantidade
aparicoes,11503
reviews,9269
reviews_em_varios_produtos,761
pares_produto_review,10605


In [26]:
perfil_posicoes = pd.DataFrame({"usuarios": quantidades_usuarios, "reviews": quantidades_reviews})
display(perfil_posicoes.value_counts().sort_index().to_frame("ocorrencias"))
assert perfil_posicoes["usuarios"].eq(perfil_posicoes["reviews"]).all()
vinculos_avaliacoes = ocorrencias_catalogo[["ocorrencia_id", "source_index", "product_id", "review_id", "user_id"]].copy()
for campo in ["review_id", "user_id"]:
    vinculos_avaliacoes[campo] = vinculos_avaliacoes[campo].str.split(",", regex=False)
vinculos_avaliacoes = vinculos_avaliacoes.explode(["review_id", "user_id"], ignore_index=True)
vinculos_avaliacoes["posicao_na_linha"] = vinculos_avaliacoes.groupby("ocorrencia_id", sort=False).cumcount() + 1
vinculos_avaliacoes = vinculos_avaliacoes[["ocorrencia_id", "source_index", "product_id", "posicao_na_linha", "review_id", "user_id"]]
display(ocorrencias_catalogo.loc[[0], ["ocorrencia_id", "user_id", "review_id"]].T.style.set_properties(**{"max-width": "650px", "white-space": "normal", "overflow-wrap": "anywhere"}))
display(vinculos_avaliacoes.loc[vinculos_avaliacoes["ocorrencia_id"].eq("OC-000001")])
autores_por_review = vinculos_avaliacoes.groupby("review_id")["user_id"].nunique()
assert autores_por_review.eq(1).all()
autoria = vinculos_avaliacoes[["review_id", "user_id"]].drop_duplicates()
avaliacoes_identificadas = avaliacoes_identificadas.merge(autoria, on="review_id", validate="one_to_one")
avaliacoes_identificadas = avaliacoes_identificadas[["review_id", "user_id", "quantidade_ocorrencias", "quantidade_produtos_associados"]]
usuarios["quantidade_reviews_distintos"] = usuarios["user_id"].map(vinculos_avaliacoes.groupby("user_id")["review_id"].nunique())
usuarios = usuarios[["user_id", "quantidade_reviews_distintos", "user_name_recuperado", "aparece_em_nome_ambiguo", "situacao_nome"]]
display(avaliacoes_identificadas.head(5))
display(usuarios.head(5))
for nome, tabela in [("usuarios", usuarios), ("avaliacoes_identificadas", avaliacoes_identificadas),
                     ("vinculos_avaliacoes", vinculos_avaliacoes)]:
    display(pd.DataFrame({"campo": tabela.columns, "tipo": tabela.dtypes.astype(str).values,
                          "nulos": tabela.isna().sum().values}).rename_axis(nome))

,,ocorrencias
usuarios,reviews,
1,1,4
2,2,11
3,3,7
4,4,9
5,5,10
6,6,6
7,7,10
8,8,1408


,0
ocorrencia_id,OC-000001
user_id,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBBSNLYT3ONILA,AHCTC6ULH4XB6YHDY6PCH2R772LQ,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,AENGU523SXMOS7JPDTW52PNNVWGQ,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ"
review_id,"R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1KD19VHEDV0OR,R3C02RMYQMK6FC,R39GQRVBUZBWGY,R2K9EDOE15QIRJ,R3OI7YT648TL8I"


,ocorrencia_id,source_index,product_id,posicao_na_linha,review_id,user_id
0,OC-000001,0,B07JW9H4J1,1,R3HXWT0LRP0NMF,AG3D6O4STAQKAY2UVGEUV46KN35Q
1,OC-000001,0,B07JW9H4J1,2,R2AJM3LFTLZHFO,AHMY5CWJMMK5BJRBBSNLYT3ONILA
2,OC-000001,0,B07JW9H4J1,3,R6AQJGUP6P86,AHCTC6ULH4XB6YHDY6PCH2R772LQ
3,OC-000001,0,B07JW9H4J1,4,R1KD19VHEDV0OR,AGYHHIERNXKA6P5T7CZLXKVPT7IQ
4,OC-000001,0,B07JW9H4J1,5,R3C02RMYQMK6FC,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q
5,OC-000001,0,B07JW9H4J1,6,R39GQRVBUZBWGY,AENGU523SXMOS7JPDTW52PNNVWGQ
6,OC-000001,0,B07JW9H4J1,7,R2K9EDOE15QIRJ,AEQJHCVTNINBS4FKTBGQRQTGTE5Q
7,OC-000001,0,B07JW9H4J1,8,R3OI7YT648TL8I,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ


,review_id,user_id,quantidade_ocorrencias,quantidade_produtos_associados
0,R100Y29EI0KGW9,AE4HN6JZ6ZJYA2ACUAOQIUXSP5FA,1,1
1,R101TILZBOMQ6F,AHHCC6KDIDI7OF6J6HV7OBGG2JGQ,1,1
2,R101VJD80D1Z15,AEMICKNJILKDILX34NH2M3J46IAQ,1,1
3,R10365HEDURWI9,AEXK37TSBFHSP2TYE63YPKETWQ7Q,3,2
4,R103G2OV6OFA3Q,AH3BI36C3I7GJFS5QMG2MKJEVQKQ,1,1


,user_id,quantidade_reviews_distintos,user_name_recuperado,aparece_em_nome_ambiguo,situacao_nome
0,AE22E2AXODSPNK3EBIHNGYS5LOSA,1,Subhajit Kansa Banik,False,Associação posicional consistente
1,AE22MK2NXQD3ZARLIOL3SLD4GU6A,1,Vishal Tiwari,False,Associação posicional consistente
2,AE22Y3KIS7SE6LI3HE2VS6WWPU4Q,1,Srinivas Pasumarthi,False,Associação posicional consistente
3,AE23RS3W7GZO7LHYKJU6KSKVM4MQ,1,Mithila Saha,False,Associação posicional consistente
4,AE23WGYTUMB5R6JJMBU4V43JIW7Q,1,Nageswararao,False,Associação posicional consistente


,campo,tipo,nulos
usuarios,,,
0,user_id,string,0
1,quantidade_reviews_distintos,int64,0
2,user_name_recuperado,string,53
3,aparece_em_nome_ambiguo,bool,0
4,situacao_nome,object,0


,campo,tipo,nulos
avaliacoes_identificadas,,,
0,review_id,object,0
1,user_id,object,0
2,quantidade_ocorrencias,int64,0
3,quantidade_produtos_associados,int64,0


,campo,tipo,nulos
vinculos_avaliacoes,,,
0,ocorrencia_id,string,0
1,source_index,Int64,0
2,product_id,string,0
3,posicao_na_linha,int64,0
4,review_id,object,0
5,user_id,object,0


### 6.3 — Investigação dos delimitadores textuais

Na primeira versão, a atribuição individual de listas maiores exigia apoio de registros unitários. Os quatro IDs unitários não reaparecem nessas listas. Ainda assim, a base contém repetições em listas e blocos distintos que podem apoiar uma hipótese. Comparamos vírgula simples, vírgula sem espaço, busca contextual e busca com evidência cruzada.

A busca mantém até 12 estados por posição e conserva quatro soluções finais. A pontuação soma sinais de fronteira sem espaço, início com maiúscula, pontuação anterior, correspondência literal em outro contexto e vocabulário pouco frequente compartilhado entre título e conteúdo. Comprimentos excessivos recebem penalidade suave. Testamos sensibilidade ao reduzir o peso da vírgula sem espaço de 4 para 2,5. A margem de dois pontos é uma regra heurística entre alternativas retidas. Pontuação, estabilidade e cobertura não medem precisão.

In [27]:
import re
from collections import Counter, defaultdict
from functools import lru_cache
from itertools import pairwise
from math import log1p

CAMPOS = ('review_title', 'review_content')
STOP = set('the and for this that with from have very good great nice best product quality money value works really was are you but not too much use using all'.split())

def partes(texto, cortes):
    return [(a+1,b,texto[a+1:b]) for a,b in pairwise((-1,*cortes,len(texto)))]

def limites(texto, sem_espaco=False):
    return tuple(m.start() for m in re.finditer(r',(?!\s)' if sem_espaco else ',',texto))

def url_spans(texto):
    # Extensões de mídia delimitam URLs do corpus mesmo sem espaço após o link.
    padrao = r'https?://[^\s<>"\']+?\.(?:jpe?g|png|webp|gif|mp4)(?=[,\s?#]|$)|https?://[^\s<>"\']+'
    return [(m.start(),m.end()) for m in re.finditer(padrao,texto,re.I)]

@lru_cache(maxsize=60000)
def palavras(texto):
    return frozenset(x for x in re.findall(r'[^\W\d_]{3,}',texto.casefold()) if x not in STOP)

def normalizar(texto):
    return ' '.join(re.findall(r'\w+',texto.casefold()))

def otimizar(texto, ids, campo, apoio, pares, raras, peso=4.0, largura=12):
    """Beam limitado por posição. Guarda alternativas, sem alegar ótimo global exato."""
    n=len(ids)
    if n==1:
        return [(0.0,())] if texto.strip() else []
    urls=url_spans(texto)
    fronteiras=[i for i in limites(texto) if not any(a<=i<b for a,b in urls)]
    if len(fronteiras)<n-1:
        return []
    sinais={}
    for i in fronteiras:
        depois=texto[i+1:].lstrip()
        sinais[i]=(peso if texto[i+1:i+2] and not texto[i+1].isspace() else 0.0)
        sinais[i]+=0.6*bool(depois and depois[0].isupper())+0.5*texto[:i].rstrip().endswith(('.', '!', '?'))
    beam=[(0.0,0,())]
    for pos,rid in enumerate(ids):
        nova=[]
        for score,inicio,cortes in beam:
            fins=[len(texto)] if pos==n-1 else fronteiras[:len(fronteiras)-(n-pos-2)]
            for fim in fins:
                if fim<=inicio: continue
                trecho=texto[inicio:fim]
                limpo=trecho.strip()
                if not limpo: continue
                minimo = -2.0 if len(limpo)==1 and not limpo.isalnum() else 0.0
                teto = 180 if campo=='review_title' else 1800
                tamanho = -0.7*log1p(max(0,len(limpo)-teto)/teto)
                cruzado = 5.0 if trecho in apoio.get(rid,()) else 0.0
                comum = palavras(trecho) & palavras(pares[pos]) & raras if pares[pos] else ()
                nota = score+sinais.get(fim,0.0)+minimo+tamanho+cruzado+0.8*min(3,len(comum))
                nova.append((nota,fim+1,cortes+(fim,) if pos<n-1 else cortes))
        beam=sorted(nova,key=lambda z:(-z[0],z[2]))[:largura]
        if not beam: break
    return [(s,c) for s,_,c in beam[:4]]

def recuperar_global(ocorrencias,vinculos,pd):
    dados=[]
    grupos=vinculos.groupby('ocorrencia_id',sort=False).review_id.agg(tuple)
    freq=Counter()
    for linha in ocorrencias.itertuples(index=False):
        ids=grupos[linha.ocorrencia_id]
        textos={c:getattr(linha,c) if pd.notna(getattr(linha,c)) else '' for c in CAMPOS}
        freq.update(palavras(textos[CAMPOS[0]]) | palavras(textos[CAMPOS[1]]))
        dados.append((linha.ocorrencia_id,linha.product_id,ids,textos))
    raras=frozenset(k for k,v in freq.items() if v<=max(3,int(len(dados)*.025)))
    sementes=defaultdict(list)
    baselines={}
    for oid,pid,ids,textos in dados:
        for campo,texto in textos.items():
            cortes=() if len(ids)==1 else limites(texto,True)
            seg=partes(texto,cortes)
            if len(seg)!=len(ids) or not all(t.strip() for _,_,t in seg): continue
            baselines[oid,campo]=seg
            for rid,(_,_,t) in zip(ids,seg):
                sementes[campo,rid].append((oid,ids,texto,t))
    evidencias=[]
    ranking=[]
    comparacao=[]
    selecionados=defaultdict(list)
    for oid,pid,ids,textos in dados:
        for campo,texto in textos.items():
            outro=CAMPOS[1-CAMPOS.index(campo)]
            alinhado=baselines.get((oid,outro),[])
            pares=[t for _,_,t in alinhado] if alinhado else ['']*len(ids)
            externas={}
            for rid in ids:
                externas[rid]=defaultdict(list)
                for outra,lista,bloco,trecho in sementes.get((campo,rid),[]):
                    if outra!=oid and lista!=ids and bloco!=texto:
                        externas[rid][trecho].append(outra)
            contextual=otimizar(texto,ids,campo,{},pares,raras)
            global_=otimizar(texto,ids,campo,externas,pares,raras)
            sensivel=otimizar(texto,ids,campo,externas,pares,raras,peso=2.5)
            b=limites(texto,True)
            marg=(global_[0][0]-global_[1][0]) if len(global_)>1 else None
            rankings=[('Virgula',[(None,limites(texto))]),('Sem espaco',[(None,b)]),
                      ('Contextual',contextual),('Global',global_),('Sensibilidade',sensivel)]
            for metodo,solucoes in rankings:
                bruto=len(solucoes[0][1])+1 if solucoes else 0
                valido=False
                for ordem,(score,cortes) in enumerate(solucoes,1):
                    seg=partes(texto,cortes)
                    assert ','.join(t for _,_,t in seg)==texto
                    if len(seg)!=len(ids): continue
                    valido=valido or all(t.strip() for _,_,t in seg)
                    for pos,(rid,(a,z,t)) in enumerate(zip(ids,seg),1):
                        apoio=externas[rid].get(t,[])
                        row=dict(review_id=rid,ocorrencia_id=oid,product_id=pid,posicao=pos,campo=campo,
                                 candidato=t,inicio=a,fim=z,metodo=metodo,ordem=ordem,score=score,
                                 esperados=len(ids),segmentos=len(seg),recomposicao_exata=True,
                                 apoio_externo='|'.join(apoio),vazio=not t.strip())
                        evidencias.append(row)
                comparacao.append((oid,campo,metodo,len(ids),bruto,valido))
            if not global_: continue
            melhor=global_[0][1]
            seg=partes(texto,melhor)
            consenso=bool(contextual and sensivel and contextual[0][1]==melhor==sensivel[0][1])
            sem_espaco=baselines.get((oid,campo))==seg
            ranking.append(dict(ocorrencia_id=oid,campo=campo,fronteiras=melhor,
                                alternativas=[c for _,c in global_],margem=marg,
                                consenso=consenso,baseline_preservado=sem_espaco))
            for pos,(rid,(inicio,fim,trecho)) in enumerate(zip(ids,seg)):
                discordantes={t for _,_,_,t in sementes.get((campo,rid),[]) if t!=trecho}
                cruzado=externas[rid].get(trecho,[])
                termos_vizinhos = set().union(*(palavras(v) for i,v in enumerate(pares) if i!=pos))
                coerencia=len((palavras(trecho)&palavras(pares[pos])&raras)-termos_vizinhos)
                if len(ids)==1:
                    status='CONFIRMADO_PELA_FONTE'
                elif discordantes:
                    status='AMBIGUO'
                elif consenso and cruzado and sem_espaco and (marg is None or marg>=2):
                    status='CONSISTENTE_MULTIPLAS_EVIDENCIAS'
                elif consenso and sem_espaco and coerencia>=2 and not url_spans(trecho) and (marg is None or marg>=2):
                    status='RECONSTRUCAO_AUTOMATICA_VALIDADA'
                elif not consenso or (marg is not None and marg<2):
                    status='AMBIGUO'
                else:
                    status='CANDIDATO_PLAUSIVEL'
                selecionados[rid,campo].append((trecho,oid,status,len(cruzado)))
    aceitos=['CONFIRMADO_PELA_FONTE','CONSISTENTE_MULTIPLAS_EVIDENCIAS','RECONSTRUCAO_AUTOMATICA_VALIDADA']
    registros=[]
    for rid in vinculos.review_id.drop_duplicates():
        row={'review_id':rid}
        metodos=[]
        for campo,sufixo in zip(CAMPOS,('titulo','conteudo')):
            opcoes=selecionados.get((rid,campo),[])
            diretas=[x for x in opcoes if x[2]==aceitos[0]]
            valores={x[0] for x in opcoes}
            ordem=sorted(opcoes,key=lambda x:(aceitos.index(x[2]) if x[2] in aceitos else 3,-x[3],x[1]))
            escolha=ordem[0] if ordem else (pd.NA,pd.NA,'NAO_RECUPERADO',0)
            trecho,oid,status,apoio=escolha
            if len({x[0] for x in diretas})>1:status='CONFLITO_CONFIRMADO'
            elif not diretas and len(valores)>1:status='DIVERGENCIA_ENTRE_OCORRENCIAS'
            aceito=status in aceitos
            row[campo]=trecho if aceito else pd.NA
            row[f'ocorrencia_{sufixo}_referencia_id']=oid if aceito else pd.NA
            row['status_recuperacao_'+sufixo]=status
            row[campo+'_candidato']=trecho if status=='CANDIDATO_PLAUSIVEL' else pd.NA
            row['quantidade_evidencias_'+sufixo]=apoio
            row['quantidade_alternativas_'+sufixo]=len(valores)
            if aceito:metodos.append(status)
        row['metodo_recuperacao']='+'.join(dict.fromkeys(metodos)) or pd.NA
        estados=[row['status_recuperacao_'+x] for x in ('titulo','conteudo')]
        row['status_consistencia']='CONFLITO_CONFIRMADO' if 'CONFLITO_CONFIRMADO' in estados else 'AMBIGUO' if 'AMBIGUO' in estados else 'EVIDENCIA_ESTRUTURAL' if metodos else 'PENDENTE'
        registros.append(row)
    rec=pd.DataFrame(registros).convert_dtypes()
    audit=pd.DataFrame(evidencias)
    audit=audit.merge(rec[['review_id','status_recuperacao_titulo','status_recuperacao_conteudo']],on='review_id',validate='many_to_one')
    audit['status_final_campo']=audit.status_recuperacao_titulo.where(audit.campo.eq(CAMPOS[0]),audit.status_recuperacao_conteudo)
    audit=audit.drop(columns=['status_recuperacao_titulo','status_recuperacao_conteudo'])
    return rec,audit,pd.DataFrame(ranking),pd.DataFrame(comparacao,columns=['ocorrencia_id','campo','metodo','esperados','segmentos','valido'])

### 6.4 — Recuperação inicial de títulos e conteúdos

As divisões convencionais são baselines. A regra sem espaço gera um índice congelado de candidatos. Na busca global, uma linha só recebe apoio de candidatos do mesmo review em outras listas e outros blocos. Assim, reproduções literais do mesmo bloco não contam como evidência cruzada. As propostas da otimização não voltam ao índice de apoio.

In [28]:
textos_recuperados, auditoria, fronteiras, comparacao = recuperar_global(ocorrencias_base,vinculos_avaliacoes,pd)
comparacao['compativel'] = comparacao.segmentos.eq(comparacao.esperados)
display(comparacao.groupby(['metodo','campo']).agg(
    ocorrencias=('campo','size'),compatibilidade_bruta=('compativel','sum'),completo_nao_vazio=('valido','sum')))

ocorrencias  compatibilidade_bruta  \
metodo        campo                                                
Contextual    review_content         1465                   1403   
              review_title           1465                   1464   
Global        review_content         1465                   1403   
              review_title           1465                   1464   
Sem espaco    review_content         1465                   1141   
              review_title           1465                   1403   
Sensibilidade review_content         1465                   1403   
              review_title           1465                   1464   
Virgula       review_content         1465                    197   
              review_title           1465                    989   

                              completo_nao_vazio  
metodo        campo                               
Contextual    review_content                1403  
              review_title                  1464  
Global        review_content                1403  
              review_title                  1464  
Sem espaco    review_content                 913  
              review_title                  1403  
Sensibilidade review_content                1403  
              review_title                  1464  
Virgula       review_content                 151  
              review_title                   989

### 6.5 — Reconstrução contextual dos casos ambíguos

O algoritmo compara divisões alternativas, conserva os offsets de origem e evita fronteiras dentro de URLs reconhecidas. Pontuação colada e URLs sem delimitador inequívoco podem dificultar a busca. A tabela mostra a distribuição da concordância entre buscas. A seguir, localizamos um caso em que alternativas permanecem próximas.

In [29]:
display(fronteiras.groupby(['campo','consenso','baseline_preservado']).size().to_frame('blocos'))
amb = fronteiras.loc[fronteiras.margem.lt(2)].sort_values('margem').head(1)
display(amb)
if not amb.empty:
    caso = amb.iloc[0]
    display(auditoria.loc[auditoria.ocorrencia_id.eq(caso.ocorrencia_id) & auditoria.campo.eq(caso.campo)
        & auditoria.metodo.eq('Global') & auditoria.ordem.le(2),
        ['ordem','posicao','review_id','inicio','fim','candidato']].assign(candidato=lambda x:x.candidato.str[:110]))

blocos
campo          consenso baseline_preservado        
review_content False    False                     6
                        True                      2
               True     False                   521
                        True                    874
review_title   True     False                    62
                        True                   1402

,ocorrencia_id,campo,fronteiras,alternativas,margem,consenso,baseline_preservado
564,OC-000289,review_content,"(71, 288, 296, 313, 763, 842, 852)","[(71, 288, 296, 313, 763, 842, 852), (234, 288...",0.0,True,False


,ordem,posicao,review_id,inicio,fim,candidato
42150,1,1,R20Y7L8T8S0B2V,0,71,",Product is Good as per the price but customer..."
42151,1,2,R19O1AZBIG1F5P,72,288,asking for money for installation and the mos...
42152,1,3,R1HA5IN5GZZEKJ,289,296,It's ok
42153,1,4,R3BGLBQWLQUBW0,297,313,Everything is ok
42154,1,5,R2GKH9JNW12AKY,314,763,"I have used Kodak 32"" led tv since last 3 year..."
42155,1,6,RKEC16QEHA2WT,764,842,Picture and sound quality is very good. Allove...
42156,1,7,R1A9NXDM3RASAL,843,852,Very nice
42157,1,8,R25TUXKCEEATJ0,853,906,"VGA port is not present, display quality is no..."
42158,2,1,R20Y7L8T8S0B2V,0,234,",Product is Good as per the price but customer..."
42159,2,2,R19O1AZBIG1F5P,235,288,if possible then start should be in minuse fo...


### 6.6 — Verificação dos reviews reutilizados

Um mesmo ID em vários produtos pode fornecer confirmação contextual ou revelar divergências. A comparação usa texto literal, uma versão normalizada somente para diagnóstico e indícios de variação por URL ou prefixo. O cadastro conserva conflitos sem escolher um texto por ordem de chegada.

A divergência metodológica é calculada por posição na mesma ocorrência. A ambiguidade exige alternativas globais próximas com trechos diferentes nessa posição. A divergência entre ocorrências compara as primeiras hipóteses globais literais em origens distintas. Só blocos unitários incompatíveis para o mesmo ID confirmam conflito neste protocolo.

In [30]:
reuso = vinculos_avaliacoes.groupby('review_id').product_id.nunique()
ids_reuso = reuso[reuso.gt(1)].index
top = auditoria.loc[auditoria.metodo.eq('Global') & auditoria.ordem.eq(1)].copy()
top['normalizado'] = top.candidato.map(normalizar)
top['sem_url'] = top.candidato.str.replace(r'https?://[^\s]+','',regex=True).map(normalizar)
variacoes = top.loc[top.review_id.isin(ids_reuso)].groupby(['review_id','campo']).agg(
    literais=('candidato','nunique'),normalizados=('normalizado','nunique'),sem_url=('sem_url','nunique')).reset_index()
def tem_prefixo(s):
    valores=sorted(set(s),key=len)
    return len(valores)>1 and any(b.startswith(a) for a,b in pairwise(valores) if a)
variacoes['prefixo'] = variacoes.set_index(['review_id','campo']).index.map(
    top.loc[top.review_id.isin(ids_reuso)].groupby(['review_id','campo']).normalizado.agg(tem_prefixo))
display(pd.Series({'IDs em vários produtos':len(ids_reuso),
    'campos com variação literal':int(variacoes.literais.gt(1).sum()),
    'variações reduzidas ao normalizar':int(variacoes.normalizados.lt(variacoes.literais).sum()),
    'variações reduzidas ao retirar URLs':int(variacoes.sem_url.lt(variacoes.normalizados).sum()),
    'indícios de prefixo':int(variacoes.prefixo.sum())}).to_frame('quantidade'))
ex = 'R1O4Z15FD40PV5'
trilha_reuso = vinculos_avaliacoes.loc[vinculos_avaliacoes.review_id.eq(ex),['ocorrencia_id','product_id','posicao_na_linha']]
for campo in CAMPOS:
    candidatos = top.loc[top.review_id.eq(ex)&top.campo.eq(campo),['ocorrencia_id','candidato']]
    trilha_reuso = trilha_reuso.merge(candidatos.rename(columns={'candidato':campo}),on='ocorrencia_id',how='left',validate='one_to_one')
    trilha_reuso[campo] = trilha_reuso[campo].str[:100]
display(trilha_reuso)
display(textos_recuperados.loc[textos_recuperados.review_id.eq(ex),['status_recuperacao_titulo','status_recuperacao_conteudo']])

,quantidade
IDs em vários produtos,761
campos com variação literal,8
variações reduzidas ao normalizar,1
variações reduzidas ao retirar URLs,7
indícios de prefixo,0


,ocorrencia_id,product_id,posicao_na_linha,review_title,review_content
0,OC-000024,B09NHVCHS9,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
1,OC-000045,B09NJN8L25,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
2,OC-000082,B09NKZXMWJ,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
3,OC-000177,B0B3N8VG24,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
4,OC-000179,B0B3MQXNFB,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
5,OC-000197,B08P9RYPLR,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
6,OC-000254,B08N1WL9XW,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
7,OC-000479,B09NHVCHS9,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
8,OC-000487,B09NL4DJ2Z,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."
9,OC-000728,B09NHVCHS9,3,Dull Physical Looks,"Look-wise, I didn’t like it. Still, I use it f..."


,status_recuperacao_titulo,status_recuperacao_conteudo
162,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL


In [31]:
def classificar_evidencias(auditoria, fronteiras, textos, pd):
    """Evidências ortogonais por review e campo, sem promover textos."""
    chave = ['review_id','campo']
    validos = auditoria.loc[~auditoria.vazio & auditoria.ordem.eq(1)]
    top = validos.loc[validos.metodo.eq('Global')]
    por_origem = ['review_id','campo','ocorrencia_id']
    metodo = validos.groupby(por_origem).candidato.nunique().gt(1).groupby(level=[0,1]).any()
    margem = fronteiras[['ocorrencia_id','campo','margem']]
    alternativas = auditoria.loc[auditoria.metodo.eq('Global')].merge(margem,on=['ocorrencia_id','campo'],validate='many_to_one')
    proximas = alternativas.loc[alternativas.margem.lt(2) & alternativas.ordem.le(2)]
    ambiguo = proximas.groupby(por_origem).candidato.nunique().gt(1).groupby(level=[0,1]).any()
    entre = top.groupby(chave).candidato.nunique().gt(1)
    confirmado = top.loc[top.esperados.eq(1)].groupby(chave).candidato.nunique().gt(1)
    sinais = pd.concat([metodo.rename('metodos'), ambiguo.rename('ambiguidade'),
                       entre.rename('entre'), confirmado.rename('confirmado')], axis=1).astype('boolean').fillna(False)
    for campo,sufixo in zip(CAMPOS,('titulo','conteudo')):
        campos = sinais.xs(campo,level='campo')
        col = 'status_recuperacao_'+sufixo
        textos['divergencia_metodos_'+sufixo] = textos.review_id.map(campos.metodos).astype('boolean').fillna(False)
        textos['ambiguidade_'+sufixo] = textos.review_id.map(campos.ambiguidade).astype('boolean').fillna(False)
        textos['divergencia_entre_ocorrencias_'+sufixo] = textos.review_id.map(campos.entre).astype('boolean').fillna(False)
        pendente = textos[campo].isna()
        textos.loc[pendente & textos['divergencia_metodos_'+sufixo],col]='DIVERGENCIA_METODOS'
        textos.loc[pendente & textos['ambiguidade_'+sufixo],col]='AMBIGUIDADE_SEGMENTACAO'
        textos.loc[pendente & textos['divergencia_entre_ocorrencias_'+sufixo],col]='DIVERGENCIA_ENTRE_OCORRENCIAS'
        textos.loc[textos.review_id.map(campos.confirmado).astype('boolean').fillna(False),col]='CONFLITO_CONFIRMADO'
    textos['divergencia_entre_ocorrencias'] = textos[['divergencia_entre_ocorrencias_titulo','divergencia_entre_ocorrencias_conteudo']].any(axis=1)
    textos['conflito_confirmado'] = textos.review_id.map(sinais.confirmado.groupby(level=0).any()).astype('boolean').fillna(False)
    prioridades = ['DIVERGENCIA_METODOS','AMBIGUIDADE_SEGMENTACAO','DIVERGENCIA_ENTRE_OCORRENCIAS','CONFLITO_CONFIRMADO']
    status=textos[['status_recuperacao_titulo','status_recuperacao_conteudo']]
    textos['status_consistencia'] = textos.status_consistencia.replace({'AMBIGUO':'PENDENTE'})
    for nome in prioridades:
        textos.loc[status.eq(nome).any(axis=1),'status_consistencia']=nome
    return textos, sinais

# Reconstrói o gatilho anterior com as sementes realmente elegíveis.
sementes = auditoria.loc[auditoria.metodo.eq('Sem espaco') & auditoria.ordem.eq(1)].copy()
validade = sementes.groupby(['ocorrencia_id','campo']).vazio.transform('any')
sementes = sementes.loc[~validade]
top = auditoria.loc[auditoria.metodo.eq('Global') & auditoria.ordem.eq(1)]
origens_metodo = sementes.merge(top[['review_id','campo','ocorrencia_id','candidato']],
    on=['review_id','campo'],suffixes=('_semente','_global'))
origens_metodo = origens_metodo.loc[origens_metodo.candidato_semente.ne(origens_metodo.candidato_global)]
multi_global = top.groupby(['review_id','campo']).candidato.nunique().gt(1)
ids_antigos = set(origens_metodo.review_id) | set(multi_global[multi_global].index.get_level_values(0))
assert len(ids_antigos)==201
textos_recuperados,sinais = classificar_evidencias(auditoria,fronteiras,textos_recuperados,pd)
diagnostico_conflitos = avaliacoes_identificadas.loc[avaliacoes_identificadas.review_id.isin(ids_antigos)].merge(
    textos_recuperados,on='review_id',validate='one_to_one')
diagnostico_conflitos['status_anterior']='CONFLITO'
diagnostico_conflitos['semente_divergiu_mesma_origem']=diagnostico_conflitos.review_id.isin(
    origens_metodo.loc[origens_metodo.ocorrencia_id_semente.eq(origens_metodo.ocorrencia_id_global),'review_id'])
diagnostico_conflitos['motivo_status_consistencia']=diagnostico_conflitos.status_consistencia.map({
    'DIVERGENCIA_METODOS':'Segmentos distintos para a mesma posição em uma ocorrência',
    'AMBIGUIDADE_SEGMENTACAO':'Alternativas próximas com fronteiras distintas',
    'DIVERGENCIA_ENTRE_OCORRENCIAS':'Primeiras hipóteses literais distintas em origens diferentes',
    'CONFLITO_CONFIRMADO':'Blocos unitários incompatíveis',
    'PENDENTE':'Hipóteses cruzadas sem confirmação nem divergência nas primeiras soluções'})
display(diagnostico_conflitos.status_consistencia.value_counts().to_frame('Dos 201 sinalizados anteriormente'))
display(pd.Series({'ocorrência única':int(diagnostico_conflitos.quantidade_ocorrencias.eq(1).sum()),
    'semente divergiu na mesma origem':int(diagnostico_conflitos.semente_divergiu_mesma_origem.sum())}).to_frame('reviews'))
colunas = ['review_id','ocorrencia_id','campo','metodo','ordem','inicio','fim','candidato','apoio_externo']
exemplos = diagnostico_conflitos.groupby('status_consistencia').head(1)
exemplos_evidencia = auditoria.loc[auditoria.review_id.isin(exemplos.review_id),colunas].merge(
    exemplos[['review_id','quantidade_ocorrencias','quantidade_produtos_associados','status_anterior','status_consistencia','motivo_status_consistencia']],
    on='review_id',validate='many_to_one')
display(exemplos_evidencia.groupby(['status_consistencia','campo']).head(2).assign(
    candidato=lambda x:x.candidato.str.slice(0,100))[['review_id','ocorrencia_id','campo','metodo','candidato','status_consistencia']])
for campo,sufixo in zip(CAMPOS,('titulo','conteudo')):
    m=textos_recuperados.set_index('review_id')['status_recuperacao_'+sufixo]
    auditoria.loc[auditoria.campo.eq(campo),'status_final_campo']=auditoria.loc[auditoria.campo.eq(campo),'review_id'].map(m)

,Dos 201 sinalizados anteriormente
status_consistencia,
DIVERGENCIA_METODOS,119
AMBIGUIDADE_SEGMENTACAO,61
DIVERGENCIA_ENTRE_OCORRENCIAS,21


,reviews
ocorrência única,164
semente divergiu na mesma origem,180


,review_id,ocorrencia_id,campo,metodo,candidato,status_consistencia
0,R1MV1NKC23DWPI,OC-000006,review_title,Virgula,Good USB cable. My experience was very good it...,DIVERGENCIA_ENTRE_OCORRENCIAS
1,R1MV1NKC23DWPI,OC-000006,review_title,Sem espaco,Good USB cable. My experience was very good it...,DIVERGENCIA_ENTRE_OCORRENCIAS
5,R1MV1NKC23DWPI,OC-000006,review_content,Virgula,https://m.media-amazon.com/images/I/51112ZRE-1...,DIVERGENCIA_ENTRE_OCORRENCIAS
6,R1MV1NKC23DWPI,OC-000006,review_content,Sem espaco,https://m.media-amazon.com/images/I/51112ZRE-1...,DIVERGENCIA_ENTRE_OCORRENCIAS
60,R11GWINZ2PW06X,OC-001010,review_title,Sem espaco,Fantastic performance and reduced boot time wi...,AMBIGUIDADE_SEGMENTACAO
61,R11GWINZ2PW06X,OC-001010,review_title,Contextual,Fantastic performance and reduced boot time wi...,AMBIGUIDADE_SEGMENTACAO
73,R11GWINZ2PW06X,OC-001010,review_content,Sem espaco,Fantastic performance and reduced boot time wi...,AMBIGUIDADE_SEGMENTACAO
74,R11GWINZ2PW06X,OC-001010,review_content,Contextual,Does not come with a screwdriver,AMBIGUIDADE_SEGMENTACAO
86,R10OPU90E2KOS8,OC-001424,review_title,Virgula,Nice and easy to use,DIVERGENCIA_METODOS
87,R10OPU90E2KOS8,OC-001424,review_title,Sem espaco,Nice and easy to use,DIVERGENCIA_METODOS


### 6.7 — Validação da recuperação textual

Conferimos offsets, ordem original e recomposição exata. Os testes negativos isolam trocas de IDs, fronteiras internas de URLs, textos iguais em IDs diferentes e conflitos de repetição. A ordem pode ser conferida contra a fonte, mas texto e pontuação não revelam sozinhos uma troca de IDs que já existisse na origem.

In [32]:
blocos = ocorrencias_base.set_index('ocorrencia_id')
mapa_posicoes = vinculos_avaliacoes.set_index(['ocorrencia_id','posicao_na_linha']).review_id
for chave,segmentos in auditoria.groupby(['ocorrencia_id','campo','metodo','ordem'],sort=False):
    oid,campo,_,_=chave
    original=blocos.at[oid,campo]
    segmentos=segmentos.sort_values('posicao')
    assert ','.join(segmentos.candidato)==original
    assert segmentos.review_id.tolist()==mapa_posicoes.loc[oid].tolist()
    assert all(original[a:b]==t for a,b,t in segmentos[['inicio','fim','candidato']].itertuples(index=False,name=None))

def micro(blocos_teste,listas_teste):
    o=pd.DataFrame([{'ocorrencia_id':str(i),'product_id':str(i),'review_title':texto,'review_content':texto}
                    for i,texto in enumerate(blocos_teste)])
    v=pd.DataFrame([(str(i),rid) for i,ids in enumerate(listas_teste) for rid in ids],
                   columns=['ocorrencia_id','review_id'])
    return recuperar_global(o,v,pd)[0].set_index('review_id')

assert micro(['Alpha phrase, beta','Different text'],[['A'],['A']]).at['A','status_recuperacao_titulo']=='CONFLITO_CONFIRMADO'
assert micro(['Same generic,Same generic'],[['A','B']]).review_title.isna().all()
assert micro(['Nice, however it broke'],[['A','B']]).review_content.isna().all()
url='https://example.org/x,y'
assert otimizar(url,('A','B'),'review_content',{},['',''],set())==[]
assert mapa_posicoes.loc['OC-000001'].tolist()!=mapa_posicoes.loc['OC-000001'].tolist()[::-1]
assert all(not (a<=c<b) for _,cortes in otimizar('https://example.org/a.jpg,Useful',('A','B'),
    'review_content',{},['',''],set()) for c in cortes for a,b in url_spans('https://example.org/a.jpg,Useful'))
def testar_sinais():
    rows = []
    def adicionar(rid,oid,metodo,trecho,ordem=1,campo='review_title',esperados=2):
        rows.append(dict(review_id=rid,ocorrencia_id=oid,campo=campo,metodo=metodo,
                         candidato=trecho,ordem=ordem,esperados=esperados,vazio=False))
    adicionar('A','1','Global','A1')
    adicionar('A','1','Sem espaco','A2')
    adicionar('B','2','Global','B1')
    adicionar('B','3','Global','B2')
    adicionar('C','4','Global','C1',campo='review_content')
    adicionar('C','4','Global','C2',ordem=2,campo='review_content')
    adicionar('D','5','Global','D1')
    adicionar('D','6','Global','D1')
    adicionar('E','7','Global','E1',esperados=1)
    adicionar('E','8','Global','E2',esperados=1)
    t=pd.DataFrame({'review_id':list('ABCDE'),'review_title':[pd.NA,pd.NA,'Título sustentado',pd.NA,pd.NA],
        'review_content':pd.NA,'status_consistencia':'PENDENTE',
        'status_recuperacao_titulo':['PENDENTE','PENDENTE','CONSISTENTE_MULTIPLAS_EVIDENCIAS','CANDIDATO_PLAUSIVEL','PENDENTE'],
        'status_recuperacao_conteudo':'PENDENTE','quantidade_alternativas_titulo':[1,2,1,1,2]})
    f=pd.DataFrame({'ocorrencia_id':['4'],'campo':['review_content'],'margem':[0.5]})
    result,_=classificar_evidencias(pd.DataFrame(rows),f,t,pd)
    z=result.set_index('review_id')
    assert z.at['A','status_consistencia']=='DIVERGENCIA_METODOS'
    assert z.at['A','quantidade_alternativas_titulo']==1 and not z.at['A','conflito_confirmado']
    assert z.at['B','status_consistencia']=='DIVERGENCIA_ENTRE_OCORRENCIAS'
    assert z.at['C','status_consistencia']=='AMBIGUIDADE_SEGMENTACAO'
    assert z.at['C','review_title']=='Título sustentado' and pd.isna(z.at['C','review_content'])
    assert z.at['D','status_recuperacao_titulo']=='CANDIDATO_PLAUSIVEL' and pd.isna(z.at['D','review_title'])
    assert z.at['E','status_consistencia']=='CONFLITO_CONFIRMADO'
    assert micro(['Uma vírgula, interna'],[['A']]).at['A','status_recuperacao_titulo']=='CONFIRMADO_PELA_FONTE'
    assert partes('Uma vírgula, interna,Outro',limites('Uma vírgula, interna,Outro',True))[0][2]=='Uma vírgula, interna'
testar_sinais()
print('Recomposição, offsets, posições e cenários sintéticos de divergência, ambiguidade e conflito aprovados.')

Recomposição, offsets, posições e cenários sintéticos de divergência, ambiguidade e conflito aprovados.


### 6.8 — Consolidação da tabela Reviews

Campos aceitos têm origem explícita. Ocorrências unitárias são evidência direta. A consistência por múltiplas evidências exige apoio literal em outro bloco e lista, concordância entre as buscas e margem mínima, sem divergência conhecida. A aceitação automática em contexto único exige, além da estabilidade, dois termos pouco frequentes em comum com o campo pareado, ausentes nas demais posições desse campo,, sem URL no trecho. Esse nível é heurístico. Os candidatos plausíveis ficam em colunas separadas. Alternativas próximas ou incompatíveis permanecem na auditoria.

In [33]:
import json
reviews=avaliacoes_identificadas.merge(textos_recuperados,on='review_id',validate='one_to_one')
reviews['hash_texto_recuperado']=reviews[list(CAMPOS)].apply(
    lambda row:hashlib.sha256(json.dumps([None if pd.isna(v) else v for v in row],ensure_ascii=False).encode()).hexdigest()
    if row.notna().any() else pd.NA,axis=1).astype('string')
for oid in ['OC-000001','OC-000364']:
    registro=blocos.loc[oid]
    display(registro[['product_id','user_id','review_id','review_title','review_content']].to_frame('origem').style.set_properties(
        **{'max-width':'680px','overflow-wrap':'anywhere','white-space':'normal','text-align':'left'}))
    exemplo=auditoria.loc[auditoria.ocorrencia_id.eq(oid)&auditoria.metodo.isin(['Sem espaco','Global'])&auditoria.ordem.eq(1)]
    display(exemplo[['review_id','metodo','posicao','campo','candidato','status_final_campo']].style.hide(axis='index').set_properties(
        **{'max-width':'320px','overflow-wrap':'anywhere','white-space':'normal'}))
    ids=mapa_posicoes.loc[oid].tolist()
    display(reviews.loc[reviews.review_id.isin(ids),['review_id','review_title','review_content',
       'status_recuperacao_titulo','status_recuperacao_conteudo']].style.hide(axis='index').set_properties(
        **{'max-width':'300px','overflow-wrap':'anywhere','white-space':'normal'}))
assert auditoria.loc[auditoria.ocorrencia_id.eq('OC-000364')&auditoria.metodo.eq('Global')&
    auditoria.campo.eq('review_title')&auditoria.ordem.eq(1)].candidato.eq(
    'Phone, camera, heating - works for me, may not for all').any()
assert auditoria.loc[auditoria.ocorrencia_id.eq('OC-000001')&auditoria.review_id.eq('R2AJM3LFTLZHFO')&
    auditoria.metodo.eq('Global')&auditoria.ordem.eq(1)].candidato.tolist()==[
    'Charging is really fast','Charging is really fast, good product.']

,origem
product_id,B07JW9H4J1
user_id,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBBSNLYT3ONILA,AHCTC6ULH4XB6YHDY6PCH2R772LQ,AGYHHIERNXKA6P5T7CZLXKVPT7IQ,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q,AENGU523SXMOS7JPDTW52PNNVWGQ,AEQJHCVTNINBS4FKTBGQRQTGTE5Q,AFC3FFC5PKFF5PMA52S3VCHOZ5FQ"
review_id,"R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1KD19VHEDV0OR,R3C02RMYQMK6FC,R39GQRVBUZBWGY,R2K9EDOE15QIRJ,R3OI7YT648TL8I"
review_title,"Satisfied,Charging is really fast,Value for money,Product review,Good quality,Good product,Good Product,As of now seems good"
review_content,"Looks durable Charging is fine tooNo complains,Charging is really fast, good product.,Till now satisfied with the quality.,This is a good product . The charging speed is slower than the original iPhone cable,Good quality, would recommend,https://m.media-amazon.com/images/W/WEBP_402378-T1/images/I/81---F1ZgHL._SY88.jpg,Product had worked well till date and was having no issue.Cable is also sturdy enough...Have asked for replacement and company is doing the same...,Value for money"


review_id,metodo,posicao,campo,candidato,status_final_campo
R3HXWT0LRP0NMF,Sem espaco,1,review_title,Satisfied,CANDIDATO_PLAUSIVEL
R2AJM3LFTLZHFO,Sem espaco,2,review_title,Charging is really fast,CANDIDATO_PLAUSIVEL
R6AQJGUP6P86,Sem espaco,3,review_title,Value for money,CANDIDATO_PLAUSIVEL
R1KD19VHEDV0OR,Sem espaco,4,review_title,Product review,CANDIDATO_PLAUSIVEL
R3C02RMYQMK6FC,Sem espaco,5,review_title,Good quality,CANDIDATO_PLAUSIVEL
R39GQRVBUZBWGY,Sem espaco,6,review_title,Good product,CANDIDATO_PLAUSIVEL
R2K9EDOE15QIRJ,Sem espaco,7,review_title,Good Product,CANDIDATO_PLAUSIVEL
R3OI7YT648TL8I,Sem espaco,8,review_title,As of now seems good,CANDIDATO_PLAUSIVEL
R3HXWT0LRP0NMF,Global,1,review_title,Satisfied,CANDIDATO_PLAUSIVEL
R2AJM3LFTLZHFO,Global,2,review_title,Charging is really fast,CANDIDATO_PLAUSIVEL


review_id,review_title,review_content,status_recuperacao_titulo,status_recuperacao_conteudo
R1KD19VHEDV0OR,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R2AJM3LFTLZHFO,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R2K9EDOE15QIRJ,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R39GQRVBUZBWGY,,,CANDIDATO_PLAUSIVEL,DIVERGENCIA_ENTRE_OCORRENCIAS
R3C02RMYQMK6FC,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R3HXWT0LRP0NMF,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R3OI7YT648TL8I,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R6AQJGUP6P86,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL


,origem
product_id,B0B4F2XCK3
user_id,"AGAELRYPMTG5SADZPDYB343EASAA,AGFN4JODOM2NTFCJQOHDBQLVDJTQ,AG7EZVSAXIVGMNDLSA55K7URQCJA,AGGF75HIEMB67OU7J3RDALBSUKQQ,AHY5CI4SU6JBYPIZ5RLAGO6W3F4A,AE4KODNBVTDCZWZO4HZM4GTRERPA,AH6HFHSYOY2OHMODD7244DHG7FUQ,AHRW5JERWYAJCZO65PDKZSOEPR6Q"
review_id,"R2K5OD0MEEBTDL,RS1N6TNO33BOK,R6KWBGOKI1N9Y,R30SKUMYLSXXDN,R1EOYHZWCRSV7B,R13JBDK4SAAYFT,RJOU5K9ECNW7Y,R2APPRANV6IERZ"
review_title,"Phone, camera, heating - works for me, may not for all,Good Mobile,Good but not excellent under this budget,Worth the price at 9499,Ok type phone... but unable to make videocall within same service provider.,Phone review,Budget king,Battery backup is good"
review_content,"I am not big on camera usage, personally. I was even mentally prepared for a bad camera, based on some reviews here. But I was pleasantly surprised that camera clicks good photos. They are not awesome, but they are decent photos that can even be shared.Now coming to my biggest grouse; heating issue. The phone started heating up while charging, but it was just a little and so I could have ignored it. But then it started heating up more and got me very concerned. I even ordered a replacement thinking I got a defective piece. But then, after further tests, I found that it is heating more when I download huge amounts of data, for example, when I restore data of my old phone, from back up. This is ok with me as, I don't perform huge data loads regularly, definitely not on phone. Then I tested by running tasks I usually perform such as checking office mails, attending office meeting on phone, watching a video from Amazon Prime, and so on. The phone did not heat up even a little. Personally, this is good for me.At this price range, this is a good phone. But if you are camera heavy user and expect to perform heavy downloads frequently, this phone may not for you. I am personally satisfied with this phone as it works for my type of usage. I will not go into plus points of this phone as they are covered by other reviews already. I am only attempting to clarify about how this phone can suit you (or not) in terms of camera and heating. I had many questions about these aspects before buying. Perhaps this review will help you make an informed decision to buy (or avoid). Cheers.,Display - BeautyCamera - decentPerformance - AmazingBattery - ok (in 5000mah u expect more tbh)Overall good phone...Also after 1day of use, i found some network connectivity issue in my jiosim, which I'm using right now in this phone, but I'll keep update this review after 1month of usage!,It's a decent mobile under this price but few things worried me , weight of the phone, too many procedure to change some settings, no screen casting. Apart from that it has good touch, a decent camera for day light , battery life is good.,I bought this smartphone for my mom. Samusung interface is very handful for easy use. Battery is superb, last whole day. Camera is mediocre but provide original colour pictures. All in all satisfied with this smartphone that i got in sale for 9499.,Unable to do video call within same service provider as in VOLTE within same service provider video call feature is available.,Product is fine. Nothing Fancy but for the budget it is a good phone.,BATTERY : more than enough for normal use Not sure in gamingCAMERA : good in this segment , can record videos in FHD 30fpsDISPLAY : since it's a LCD display the quality is a bit less , but goodV RAM : you can add upto 2gb of virtual ram but have to sacrifice your storage Space to use it OVERALL A GOOD BUDGET PHONE,Finger print is working speedy battery backup is good camera quality is also good"


review_id,metodo,posicao,campo,candidato,status_final_campo
R2K5OD0MEEBTDL,Sem espaco,1,review_title,"Phone, camera, heating - works for me, may not for all",CANDIDATO_PLAUSIVEL
RS1N6TNO33BOK,Sem espaco,2,review_title,Good Mobile,CANDIDATO_PLAUSIVEL
R6KWBGOKI1N9Y,Sem espaco,3,review_title,Good but not excellent under this budget,CANDIDATO_PLAUSIVEL
R30SKUMYLSXXDN,Sem espaco,4,review_title,Worth the price at 9499,CANDIDATO_PLAUSIVEL
R1EOYHZWCRSV7B,Sem espaco,5,review_title,Ok type phone... but unable to make videocall within same service provider.,RECONSTRUCAO_AUTOMATICA_VALIDADA
R13JBDK4SAAYFT,Sem espaco,6,review_title,Phone review,CANDIDATO_PLAUSIVEL
RJOU5K9ECNW7Y,Sem espaco,7,review_title,Budget king,CANDIDATO_PLAUSIVEL
R2APPRANV6IERZ,Sem espaco,8,review_title,Battery backup is good,CANDIDATO_PLAUSIVEL
R2K5OD0MEEBTDL,Global,1,review_title,"Phone, camera, heating - works for me, may not for all",CANDIDATO_PLAUSIVEL
RS1N6TNO33BOK,Global,2,review_title,Good Mobile,CANDIDATO_PLAUSIVEL


review_id,review_title,review_content,status_recuperacao_titulo,status_recuperacao_conteudo
R13JBDK4SAAYFT,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R1EOYHZWCRSV7B,Ok type phone... but unable to make videocall within same service provider.,Unable to do video call within same service provider as in VOLTE within same service provider video call feature is available.,RECONSTRUCAO_AUTOMATICA_VALIDADA,RECONSTRUCAO_AUTOMATICA_VALIDADA
R2APPRANV6IERZ,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R2K5OD0MEEBTDL,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R30SKUMYLSXXDN,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
R6KWBGOKI1N9Y,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
RJOU5K9ECNW7Y,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL
RS1N6TNO33BOK,,,CANDIDATO_PLAUSIVEL,CANDIDATO_PLAUSIVEL


### 6.9 — Preparação para classificação temática e sentimental

O filtro de maior confiança exige ambos os campos confirmados pela fonte ou consistentes em múltiplas evidências. A recuperação automática em contexto único forma um conjunto exploratório, separado do conjunto elegível. O controle de dependências continua necessário. Nenhum grupo possui rótulos humanos nesta etapa.

A separação dos sinais de incerteza conserva os requisitos de aceitação. A exclusão por conflito exige incompatibilidade confirmada. A trilha de blocos integrais usa os 1.212 grupos exatos e mantém as associações a ocorrências e produtos, sem atribuir cada bloco a um usuário.

In [34]:
altos=['CONFIRMADO_PELA_FONTE','CONSISTENTE_MULTIPLAS_EVIDENCIAS']
reviews['elegivel_ml_textual']=reviews.status_recuperacao_titulo.isin(altos)&reviews.status_recuperacao_conteudo.isin(altos)
reviews['elegivel_ml_textual'] &= ~reviews.conflito_confirmado
reviews['grupo_uso_textual']='PENDENTE'
reviews.loc[reviews[list(CAMPOS)].notna().all(axis=1),'grupo_uso_textual']='EXPLORATORIO'
reviews.loc[reviews.elegivel_ml_textual,'grupo_uso_textual']='MAIOR_CONFIANCA'
reviews.loc[reviews.conflito_confirmado,'grupo_uso_textual']='EXCLUIDO_CONFLITO'
display(reviews.grupo_uso_textual.value_counts().to_frame('reviews'))
display(reviews.loc[reviews[list(CAMPOS)].notna().all(axis=1),['review_id','review_title','review_content','grupo_uso_textual']].head(6)
    .style.hide(axis='index').set_properties(**{'max-width':'450px','white-space':'normal','overflow-wrap':'anywhere'}))
display(pd.DataFrame({'coluna':reviews.columns,'tipo':reviews.dtypes.astype(str).values,'nulos':reviews.isna().sum().values}))

,reviews
grupo_uso_textual,
PENDENTE,9098
EXPLORATORIO,101
MAIOR_CONFIANCA,70


review_id,review_title,review_content,grupo_uso_textual
R11D3U0V2XKDKF,Good,Good,MAIOR_CONFIANCA
R12N4I2XRPP114,"Really good product, Screws are available under lower thermocol","People who are saying that screws didn't come. Please check under lower thermocol. Screws are there. I raised return request bought new Morphy richards from store and found this thing. So cancelled the return request. I got 2 now because of this issue. Why Morphy richards? Why hiding it away where people are not able to find it?Overall product is really good, please go for it.",EXPLORATORIO
R13082370PJO1Z,Not supporting for CarPlay,Not supporting for CarPlay. And this product I want to cancel but Amazon don’t have option to cancel.,EXPLORATORIO
R14ALM4LONM07K,Some improvement required,1. Please add app gallery when we do right to left swap on home screen. Because right to left swap or left to right swap have same pages.2. It should ring instead of vibrate in alarm or you can give it optional depends on user.3. The temperature doesn't sync fast I have to go in the noisefit app again and again and do it on and off in weather settings to sync feels like jugad. All these things can be fix from updates so pls try to do it,MAIOR_CONFIANCA
R16MDWVEULVTGY,उपयोगी एवं संतोषजनक,हार्ट रेट एवं ऑक्सीजन,MAIOR_CONFIANCA
R17U7AO7GNBOX8,It is a good watch,It's a good watch but the strap always comes out,MAIOR_CONFIANCA


,coluna,tipo,nulos
0,review_id,object,0
1,user_id,object,0
2,quantidade_ocorrencias,int64,0
3,quantidade_produtos_associados,int64,0
4,review_title,string,9062
5,ocorrencia_titulo_referencia_id,string,9062
6,status_recuperacao_titulo,string,0
7,review_title_candidato,string,624
8,quantidade_evidencias_titulo,Int64,0
9,quantidade_alternativas_titulo,Int64,0


## 7 — Construindo e validando os relacionamentos

### 7.1 — Construção de Vínculos

A ponte retém uma linha por ocorrência e posição. Produto e usuário na ponte são redundâncias verificadas.

In [35]:
vinculos_reviews = vinculos_avaliacoes.copy()
display(vinculos_reviews.head())

,ocorrencia_id,source_index,product_id,posicao_na_linha,review_id,user_id
0,OC-000001,0,B07JW9H4J1,1,R3HXWT0LRP0NMF,AG3D6O4STAQKAY2UVGEUV46KN35Q
1,OC-000001,0,B07JW9H4J1,2,R2AJM3LFTLZHFO,AHMY5CWJMMK5BJRBBSNLYT3ONILA
2,OC-000001,0,B07JW9H4J1,3,R6AQJGUP6P86,AHCTC6ULH4XB6YHDY6PCH2R772LQ
3,OC-000001,0,B07JW9H4J1,4,R1KD19VHEDV0OR,AGYHHIERNXKA6P5T7CZLXKVPT7IQ
4,OC-000001,0,B07JW9H4J1,5,R3C02RMYQMK6FC,AG4OGOFWXJZTQ2HKYIOCOY3KXF2Q


### 7.2 — Integridade referencial

As chaves e as referências declaradas são verificadas em memória e após a releitura.

### 7.3 — Testes de navegação entre entidades

Seguimos uma categoria até seus produtos e um review reutilizado até os produtos e o usuário associado.

In [36]:
categoria = produtos.loc[produtos.product_id.eq('B09MT84WV5'),'categoria_id'].iloc[0]
recorte = produtos.loc[produtos.categoria_id.eq(categoria)]
assert set(recorte.product_id)==set(fonte.loc[fonte.category.eq(recorte.category_original.iloc[0]),'product_id'])
trilha = vinculos_reviews.loc[vinculos_reviews.review_id.eq('R1O4Z15FD40PV5')].merge(usuarios[['user_id','situacao_nome']],on='user_id',validate='many_to_one')
display(trilha)
display(recorte[['product_id','discounted_price_num']].head())

,ocorrencia_id,source_index,product_id,posicao_na_linha,review_id,user_id,situacao_nome
0,OC-000024,23,B09NHVCHS9,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
1,OC-000045,44,B09NJN8L25,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
2,OC-000082,81,B09NKZXMWJ,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
3,OC-000177,176,B0B3N8VG24,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
4,OC-000179,178,B0B3MQXNFB,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
5,OC-000197,196,B08P9RYPLR,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
6,OC-000254,253,B08N1WL9XW,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
7,OC-000479,478,B09NHVCHS9,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
8,OC-000487,486,B09NL4DJ2Z,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente
9,OC-000728,727,B09NHVCHS9,3,R1O4Z15FD40PV5,AEBWA5I4QFCA3P3OBEPMELBGN4GQ,Associação posicional consistente


,product_id,discounted_price_num
297,B07DJLFMPS,369.0
671,B08L5FM4JC,649.0
672,B08L5HMJVW,369.0
971,B09MT6XSFW,599.0
972,B09MT84WV5,1149.0


### 7.4 — Conferência de granularidade

As medidas comerciais permanecem no cadastro de produtos. Contagens textuais distinguem review único, associação produto–review e aparição posicional.

In [37]:
display(pd.Series({'produtos':len(produtos),'reviews distintos':len(reviews),
    'pares produto–review':len(vinculos_reviews[['product_id','review_id']].drop_duplicates()),
    'posições':len(vinculos_reviews)}).to_frame('quantidade'))

,quantidade
produtos,1351
reviews distintos,9269
pares produto–review,10605
posições,11503


### 7.5 — Diagrama ER físico final

O diagrama completo é gerado a partir do esquema exportado. As arestas opcionais de origem textual são distintas da relação de aparição na ponte.

In [38]:
from IPython.display import Markdown
import tempfile

ocorrencias_base['grupo_textual_id'] = 'TXT-' + (
    ocorrencias_base.groupby(['review_title','review_content'],dropna=False).source_index.transform('min')+1
).astype('string').str.zfill(6)
chaves = {'ocorrencias_base':['ocorrencia_id'],'categorias':['categoria_id'],
          'produtos':['product_id'],'usuarios':['user_id'],'reviews':['review_id'],
          'vinculos_reviews':['ocorrencia_id','posicao_na_linha']}
fks = [
    ('produtos','categoria_id','categorias','categoria_id',False),
    ('ocorrencias_base','product_id','produtos','product_id',False),
    ('produtos','ocorrencia_id_referencia','ocorrencias_base','ocorrencia_id',False),
    ('reviews','user_id','usuarios','user_id',False),
    ('vinculos_reviews','ocorrencia_id','ocorrencias_base','ocorrencia_id',False),
    ('vinculos_reviews','review_id','reviews','review_id',False),
    ('vinculos_reviews','product_id','produtos','product_id',False),
    ('vinculos_reviews','user_id','usuarios','user_id',False),
    ('reviews','ocorrencia_titulo_referencia_id','ocorrencias_base','ocorrencia_id',True),
    ('reviews','ocorrencia_conteudo_referencia_id','ocorrencias_base','ocorrencia_id',True),
]
contrato_ml = dict(classificacao_id='string',review_id='string',assunto='string',polaridade='string',
    trecho_evidencia='string',score_modelo='Float64',metodo='string',versao_modelo='string',status_validacao='string')

def modelo():
    cadastro = produtos.drop(columns='category_original').rename(columns={
        'about_product_referencia':'about_product','product_link_referencia':'product_link',
        'img_link_referencia':'img_link','quantidade_observacoes':'quantidade_ocorrencias_origem'})
    tabelas = dict(ocorrencias_base=ocorrencias_base,categorias=categorias,produtos=cadastro,
                   usuarios=usuarios,reviews=reviews,vinculos_reviews=vinculos_reviews)
    return {k:t.convert_dtypes(convert_integer=False) for k,t in tabelas.items()}

def validar(t):
    testes = []
    for nome,pk in chaves.items():
        assert t[nome][pk].notna().all().all() and not t[nome].duplicated(pk).any(), nome
        testes.append('PK '+nome)
    for origem,campo,destino,chave,opcional in fks:
        s = t[origem][campo]
        assert (opcional or s.notna().all()) and s.dropna().isin(t[destino][chave]).all(), (origem,campo)
        testes.append(f'FK {origem}.{campo}')
    o,c,p,u,r,v = (t[k] for k in chaves)
    pd.testing.assert_frame_equal(o[fonte.columns],fonte)
    assert o.source_index.tolist()==list(range(len(fonte)))
    assert o.ocorrencia_id.tolist()==[f'OC-{i+1:06}' for i in range(len(fonte))]
    assert len(c)==211 and c.categoria_macro_original.nunique()==9
    assert c[niveis].apply(lambda x:'|'.join(x.dropna()),axis=1).tolist()==c.category_original.tolist()
    ref = p.ocorrencia_id_referencia.map(o.set_index('ocorrencia_id').source_index)
    assert ref.tolist()==p.source_index_referencia.tolist()
    assert p.categoria_id.map(c.set_index('categoria_id').category_original).tolist()==ref.map(o.set_index('source_index').category).tolist()
    for campo in campos_comerciais:
        assert p[campo].equals(ref.map(observacoes.set_index('source_index')[campo]))
    assert set(p.product_id)==set(o.product_id)
    assert set(u.user_id)==set(v.user_id) and set(r.review_id)==set(v.review_id)
    assert len(v)==11503
    s = v.sort_values(['ocorrencia_id','posicao_na_linha'])
    assert s.posicao_na_linha.tolist()==(s.groupby('ocorrencia_id').cumcount()+1).tolist()
    for campo in ['review_id','user_id']:
        assert s.groupby('ocorrencia_id')[campo].agg(','.join).tolist()==o.sort_values('ocorrencia_id')[campo].tolist()
    for campo in ['product_id','source_index']:
        assert v[campo].tolist()==v.ocorrencia_id.map(o.set_index('ocorrencia_id')[campo]).tolist()
    assert p.product_id.tolist()==p.ocorrencia_id_referencia.map(o.set_index('ocorrencia_id').product_id).tolist()
    assert v.user_id.tolist()==v.review_id.map(r.set_index('review_id').user_id).tolist()
    assert r.quantidade_ocorrencias.tolist()==r.review_id.map(v.review_id.value_counts()).tolist()
    assert r.quantidade_produtos_associados.tolist()==r.review_id.map(v.groupby('review_id').product_id.nunique()).tolist()
    assert u.quantidade_reviews_distintos.tolist()==u.user_id.map(v.groupby('user_id').review_id.nunique()).tolist()
    assert o.grupo_textual_id.nunique()==1212
    assert c[['categoria_macro_ml','metodo_macro_ml','versao_macro_ml']].isna().all().all()
    assert set(r.loc[r.elegivel_ml_textual,'review_id']) <= set(r.dropna(subset=['review_title','review_content']).review_id)
    pares = set(zip(v.ocorrencia_id,v.review_id))
    for campo,sufixo in [('review_title','titulo'),('review_content','conteudo')]:
        ref_col='ocorrencia_'+sufixo+'_referencia_id'
        assert r[campo].notna().equals(r[ref_col].notna())
        orig=o.set_index('ocorrencia_id')[campo]
        for rid,oid,valor in r.loc[r[campo].notna(),['review_id',ref_col,campo]].itertuples(index=False,name=None):
            assert (oid,rid) in pares and valor in orig[oid]
    testes += ['origem literal e posição','hierarquia e referências comerciais','listas recompostas e posições',
               'redundâncias reconciliadas','contagens por entidade','grupos exatos','contrato ML sem previsões','origens textuais']
    return testes

def comparar_oficial():
    """Reconcilia o modelo gerado com a versão oficial, no esquema canônico."""
    for nome,tabela in modelo().items():
        oficial = pd.read_csv(pasta_oficial/f'{nome}.csv',dtype=tabela.dtypes.to_dict(),
            encoding='utf-8-sig',keep_default_na=False,na_values=['__AUSENTE__'],float_precision='round_trip')
        pd.testing.assert_frame_equal(tabela,oficial,check_exact=True)
    print('Seis entidades reconciliadas com a publicação oficial.')

def er(t):
    linhas = ['erDiagram']
    for o,c,d,k,op in fks:
        linhas.append(f'    {d.upper()} {"|o" if op else "||"}--o{{ {o.upper()} : {c}')
    linhas.append('    REVIEWS ||--o{ CLASSIFICACOES_ML_REVIEWS : contrato_futuro')
    for nome,esquema in {**{k:{c:str(d) for c,d in t[k].dtypes.items()} for k in t},
                         'classificacoes_ml_reviews':contrato_ml}.items():
        linhas.append(f'    {nome.upper()} {{')
        for campo,tipo in esquema.items():
            marcas = []
            if campo in chaves.get(nome,[]) or campo=='classificacao_id': marcas.append('PK')
            if any(o==nome and c==campo for o,c,*_ in fks) or (nome=='classificacoes_ml_reviews' and campo=='review_id'): marcas.append('FK')
            linhas.append(f'        {tipo} {campo} '+','.join(marcas))
        linhas.append('    }')
    return '\n'.join(linhas)

def exportar():
    assert PUBLICAR_CSVS is False
    t = modelo()
    comparar_oficial()
    testes = validar(t)
    destino = Path(tempfile.mkdtemp(prefix='proposta_02_',dir=os.environ.get('AMAZON_02_STAGING')))
    relidas = {}
    for nome,dados in t.items():
        marcador = '__AUSENTE__'
        assert not any(dados[c].eq(marcador).any() for c in dados.select_dtypes('string'))
        p = destino/f'{nome}.csv'
        dados.to_csv(p,index=False,encoding='utf-8-sig',na_rep=marcador)
        relidas[nome] = pd.read_csv(p,dtype=dados.dtypes.to_dict(),encoding='utf-8-sig',
            keep_default_na=False,na_values=[marcador],float_precision='round_trip')
        pd.testing.assert_frame_equal(dados,relidas[nome],check_exact=True)
    assert validar(relidas)==testes
    assert hashes_oficiais_antes=={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in pasta_oficial.glob('*.csv')}
    (destino/'modelo_er_fisico_completo.md').write_text('# Modelo físico e contrato futuro de ML\n\nOs tipos refletem o esquema da exportação. As referências de título e conteúdo são opcionais. As arestas de origem servem à auditoria. A tabela de classificações é contrato futuro.\n\n```mermaid\n'+er(t)+'\n```\n',encoding='utf-8')
    info = {k:{'linhas':len(d),'chave':chaves[k],'tipos':{c:str(dt) for c,dt in d.dtypes.items()},
               'nulos':d.isna().sum().to_dict()} for k,d in t.items()}
    tem_t=reviews.review_title.notna()
    tem_c=reviews.review_content.notna()
    cobertura={
        'reviews':len(reviews),'titulo':int(tem_t.sum()),'conteudo':int(tem_c.sum()),
        'ambos':int((tem_t&tem_c).sum()),'parcial':int((tem_t^tem_c).sum()),
        'sem_texto':int((~tem_t&~tem_c).sum()),
        'titulo_candidato':int(reviews.review_title_candidato.notna().sum()),
        'conteudo_candidato':int(reviews.review_content_candidato.notna().sum()),
        'evidencia_cruzada':int(auditoria.loc[auditoria.metodo.eq('Global')&auditoria.ordem.eq(1)&auditoria.apoio_externo.ne(''),'review_id'].nunique()),
        'ambiguos':int(reviews.status_consistencia.eq('AMBIGUO').sum()),
        'conflitos':int(reviews.status_consistencia.eq('CONFLITO').sum()),
        'elegiveis':int(reviews.elegivel_ml_textual.sum())}
    cobertura.update({
        'maior_confianca':int(reviews.grupo_uso_textual.eq('MAIOR_CONFIANCA').sum()),
        'exploratorio':int(reviews.grupo_uso_textual.eq('EXPLORATORIO').sum()),
        'divergencia_metodos':int(reviews[['divergencia_metodos_titulo','divergencia_metodos_conteudo']].any(axis=1).sum()),
        'ambiguidade_segmentacao':int(reviews[['ambiguidade_titulo','ambiguidade_conteudo']].any(axis=1).sum()),
        'divergencia_entre_ocorrencias':int(reviews.divergencia_entre_ocorrencias.sum()),
        'conflitos_confirmados':int(reviews.conflito_confirmado.sum()),
        'ambiguos':int(reviews.status_consistencia.eq('AMBIGUIDADE_SEGMENTACAO').sum())})
    anterior = {'reviews':9269,'titulo':207,'conteudo':210,'ambos':171,'parcial':75,
        'sem_texto':9023,'elegiveis':70,'maior_confianca':70,'exploratorio':101,'conflitos':201}
    diagnostico_conflitos.to_csv(destino/'diagnostico_conflitos_anteriores.csv',index=False,encoding='utf-8-sig')
    exemplos_evidencia.to_csv(destino/'exemplos_evidencias_reviews.csv',index=False,encoding='utf-8-sig')
    comparativo = pd.DataFrame({'antes':anterior,'depois':cobertura})
    comparativo['depois_pct']=comparativo.depois.div(len(reviews)).mul(100).round(2)
    display(comparativo)
    (destino/'validacao.json').write_text(json.dumps(dict(entidades=info,testes=testes,cobertura=cobertura,
        fks=fks,contrato_ml=contrato_ml),ensure_ascii=False,indent=2),encoding='utf-8')
    auditoria.to_csv(destino/'auditoria_candidatos_reviews.csv.gz',index=False,encoding='utf-8-sig',compression='gzip')
    comparacao.to_csv(destino/'comparacao_segmentacao.csv',index=False,encoding='utf-8-sig')
    fronteiras.to_json(destino/'fronteiras_reviews.jsonl',orient='records',lines=True,force_ascii=False)
    variacoes.to_csv(destino/'variacoes_reviews_reutilizados.csv',index=False,encoding='utf-8-sig')
    pool=auditoria.loc[auditoria.metodo.eq('Global')&auditoria.ordem.eq(1)].copy()
    pool=pool.merge(fronteiras[['ocorrencia_id','campo','consenso','margem','baseline_preservado']],
                    on=['ocorrencia_id','campo'],validate='many_to_one')
    pool['original']=[blocos.at[oid,c] for oid,c in zip(pool.ocorrencia_id,pool.campo)]
    estratos={
        'unitária':pool.esperados.eq(1),'oito ou mais':pool.esperados.ge(8),
        'vírgula interna no título':pool.campo.eq('review_title')&pool.candidato.str.contains(','),
        'conteúdo longo':pool.campo.eq('review_content')&pool.original.str.len().gt(3000),
        'URL':pool.original.str.contains('http',regex=False),'review reutilizado':pool.review_id.isin(ids_reuso),
        'apoio entre contextos':pool.apoio_externo.ne(''),'conflitante':pool.status_final_campo.isin(['CONFLITO_CONFIRMADO','DIVERGENCIA_ENTRE_OCORRENCIAS']),
        'otimização mudou a baseline':~pool.baseline_preservado,
        'ambíguo':pool.status_final_campo.eq('AMBIGUO')}
    amostras=[pool.loc[mask].sample(n=min(5,int(mask.sum())),random_state=42).assign(estrato=nome)
              for nome,mask in estratos.items()]
    revisao=pd.concat(amostras)
    sem_solucao=comparacao.loc[comparacao.metodo.eq('Global')&~comparacao.valido].head(5)
    pendentes=sem_solucao.assign(estrato='sem solução completa',
        original=[blocos.at[oid,c] for oid,c in zip(sem_solucao.ocorrencia_id,sem_solucao.campo)])
    revisao=pd.concat([revisao,pendentes],ignore_index=True)
    revisao['ids_originais']=revisao.ocorrencia_id.map(blocos.review_id)
    revisao['revisao_humana']='PENDENTE'
    revisao['resultado_revisao']=pd.NA
    revisao['observacao_revisor']=pd.NA
    revisao.to_csv(destino/'amostra_revisao_textual.csv',index=False,encoding='utf-8-sig')
    comparativo.to_csv(destino/'antes_depois.csv',encoding='utf-8-sig')
    print('Proposta validada em',destino)

tabelas = modelo()
display(Markdown('```mermaid\n'+er(tabelas)+'\n```'))

```mermaid
erDiagram
    CATEGORIAS ||--o{ PRODUTOS : categoria_id
    PRODUTOS ||--o{ OCORRENCIAS_BASE : product_id
    OCORRENCIAS_BASE ||--o{ PRODUTOS : ocorrencia_id_referencia
    USUARIOS ||--o{ REVIEWS : user_id
    OCORRENCIAS_BASE ||--o{ VINCULOS_REVIEWS : ocorrencia_id
    REVIEWS ||--o{ VINCULOS_REVIEWS : review_id
    PRODUTOS ||--o{ VINCULOS_REVIEWS : product_id
    USUARIOS ||--o{ VINCULOS_REVIEWS : user_id
    OCORRENCIAS_BASE |o--o{ REVIEWS : ocorrencia_titulo_referencia_id
    OCORRENCIAS_BASE |o--o{ REVIEWS : ocorrencia_conteudo_referencia_id
    REVIEWS ||--o{ CLASSIFICACOES_ML_REVIEWS : contrato_futuro
    OCORRENCIAS_BASE {
        string ocorrencia_id PK
        Int64 source_index 
        string product_id FK
        string product_name 
        string category 
        string discounted_price 
        string actual_price 
        string discount_percentage 
        string rating 
        string rating_count 
        string about_product 
        string user_id 
        string user_name 
        string review_id 
        string review_title 
        string review_content 
        string img_link 
        string product_link 
        string grupo_textual_id 
    }
    CATEGORIAS {
        string categoria_id PK
        string category_original 
        string categoria_nivel_1 
        string categoria_nivel_2 
        string categoria_nivel_3 
        string categoria_nivel_4 
        string categoria_nivel_5 
        string categoria_nivel_6 
        string categoria_nivel_7 
        string categoria_macro_original 
        string categoria_especifica 
        Int64 profundidade_categoria 
        string categoria_macro_ml 
        string metodo_macro_ml 
        string versao_macro_ml 
        string status_macro_ml 
    }
    PRODUTOS {
        string product_id PK
        string product_name 
        string about_product 
        string product_link 
        string img_link 
        Int64 source_index_referencia 
        Float64 actual_price_num 
        Float64 discounted_price_num 
        Float64 discount_percentage_num 
        Float64 rating_num 
        Int64 rating_count_num 
        Float64 rating_analitico_num 
        boolean rating_estimado 
        string metodo_estimativa_rating 
        Int64 rating_count_analitico_num 
        boolean rating_count_estimado 
        string metodo_estimativa_rating_count 
        string rating_count_situacao 
        Int64 quantidade_ocorrencias_origem 
        boolean conflito_preco_original 
        boolean conflito_preco_desconto 
        boolean conflito_percentual 
        boolean conflito_contagem 
        boolean conflito_descricao 
        boolean conflito_imagem 
        string criterio_selecao_referencia 
        string ocorrencia_id_referencia FK
        string categoria_id FK
    }
    USUARIOS {
        string user_id PK
        int64 quantidade_reviews_distintos 
        string user_name_recuperado 
        boolean aparece_em_nome_ambiguo 
        string situacao_nome 
    }
    REVIEWS {
        string review_id PK
        string user_id FK
        int64 quantidade_ocorrencias 
        int64 quantidade_produtos_associados 
        string review_title 
        string ocorrencia_titulo_referencia_id FK
        string status_recuperacao_titulo 
        string review_title_candidato 
        Int64 quantidade_evidencias_titulo 
        Int64 quantidade_alternativas_titulo 
        string review_content 
        string ocorrencia_conteudo_referencia_id FK
        string status_recuperacao_conteudo 
        string review_content_candidato 
        Int64 quantidade_evidencias_conteudo 
        Int64 quantidade_alternativas_conteudo 
        string metodo_recuperacao 
        string status_consistencia 
        boolean divergencia_metodos_titulo 
        boolean ambiguidade_titulo 
        boolean divergencia_entre_ocorrencias_titulo 
        boolean divergencia_metodos_conteudo 
        boolean ambiguidade_conteudo 
        boolean divergencia_entre_ocorrencias_conteudo 
        boolean divergencia_entre_ocorrencias 
        boolean conflito_confirmado 
        string hash_texto_recuperado 
        boolean elegivel_ml_textual 
        string grupo_uso_textual 
    }
    VINCULOS_REVIEWS {
        string ocorrencia_id PK,FK
        Int64 source_index 
        string product_id FK
        int64 posicao_na_linha PK
        string review_id FK
        string user_id FK
    }
    CLASSIFICACOES_ML_REVIEWS {
        string classificacao_id PK
        string review_id FK
        string assunto 
        string polaridade 
        string trecho_evidencia 
        Float64 score_modelo 
        string metodo 
        string versao_modelo 
        string status_validacao 
    }
```

## 8 — Qualidade final, exportação e continuidade

### 8.1 — Auditoria geral das entidades

Os esquemas registram tipos, grãos, chaves e ausências. A contagem de colunas inclui campos de proveniência e cenários.

In [39]:
display(pd.DataFrame([{'entidade':k,'linhas':len(t),'colunas':len(t.columns),'chave':', '.join(chaves[k])} for k,t in tabelas.items()]))

,entidade,linhas,colunas,chave
0,ocorrencias_base,1465,19,ocorrencia_id
1,categorias,211,16,categoria_id
2,produtos,1351,28,product_id
3,usuarios,9050,5,user_id
4,reviews,9269,29,review_id
5,vinculos_reviews,11503,6,"ocorrencia_id, posicao_na_linha"


### 8.2 — Comparação com o Notebook 01

Recalculamos contagens da origem e dos vínculos para reconciliar as unidades da investigação estrutural.

In [40]:
indicadores = [len(fonte),fonte.shape[1],len(produtos),int(frequencias_produtos.gt(1).sum()),
    int(frequencias_produtos[frequencias_produtos.gt(1)].sum()),len(fonte)-len(produtos),int(fonte.duplicated().sum()),
    len(categorias),categorias.profundidade_categoria.min(),categorias.profundidade_categoria.max(),
    categorias.categoria_macro_original.nunique(),len(usuarios),len(reviews),len(vinculos_reviews),
    len(fonte[['review_title','review_content']].drop_duplicates()),int(observacoes.rating_invalido.sum()),int(fonte.rating_count.isna().sum())]
assert indicadores == [1465,16,1351,92,206,114,0,211,2,7,9,9050,9269,11503,1212,1,2]
display(pd.DataFrame({'medida':['ocorrências','campos originais','produtos','IDs repetidos','linhas desses IDs','excedentes','duplicatas','caminhos','profundidade mínima','profundidade máxima','macros','usuários','reviews','vínculos','grupos textuais','nota inválida','contagens ausentes'],'resultado':indicadores}))

,medida,resultado
0,ocorrências,1465
1,campos originais,16
2,produtos,1351
3,IDs repetidos,92
4,linhas desses IDs,206
5,excedentes,114
6,duplicatas,0
7,caminhos,211
8,profundidade mínima,2
9,profundidade máxima,7


### 8.3 — Comparação com o tratamento anterior

A dimensão de categorias substitui atributos hierárquicos repetidos no cadastro. A ponte e os valores comerciais observados reconciliam com a base oficial. As novas colunas de Reviews acrescentam qualidade e proveniência.

In [41]:
print('Comparação oficial após montar os cenários, na exportação.')

Contratos legados reconciliados (observado)


### 8.4 — Tratamentos complementares e sensibilidade

Os cenários opcionais mantêm os métodos anteriores e são transferidos à ocorrência selecionada depois da seleção comercial. O observado permanece a base dos indicadores.

In [42]:
excecoes_rating = observacoes.loc[observacoes["rating_invalido"]]
notas_mesmo_id = observacoes.loc[observacoes["product_id"].isin(excecoes_rating["product_id"])
                                & observacoes["rating_num"].notna()]
display(excecoes_rating[["source_index", "product_id", "category", "rating", "rating_num"]].T.style.set_properties(**{"max-width": "600px", "white-space": "normal", "overflow-wrap": "anywhere"}))
display(notas_mesmo_id[["product_id", "rating_num"]])
# O subconjunto serve apenas à comparação de notas; observacoes mantém todas as linhas.
notas_comparacao = observacoes.loc[observacoes["rating_num"].notna(),
                                   ["product_id", "category", "rating_num"]]
notas_por_id = notas_comparacao.groupby("product_id")["rating_num"].nunique()
assert notas_por_id.gt(1).sum() == 0
notas_comparacao = notas_comparacao.loc[
    notas_comparacao["product_id"].isin(notas_por_id.loc[notas_por_id.eq(1)].index)
]
comparacao_rating = None
minimo_comparaveis = 10

# A investigação individual exige uma só exceção e uma categoria com nível superior.
if len(excecoes_rating) == 1 and notas_mesmo_id.empty and excecoes_rating["category"].str.contains("|", regex=False, na=False).all():
    categoria_especifica = excecoes_rating["category"].item()
    categoria_superior = categoria_especifica.rsplit("|", 1)[0]
    grupo_especifico = notas_comparacao.loc[notas_comparacao["category"].eq(categoria_especifica)]
    grupo_superior = notas_comparacao.loc[
        notas_comparacao["category"].eq(categoria_superior)
        | notas_comparacao["category"].str.startswith(categoria_superior + "|", na=False)
    ]
    comparacao_rating = pd.concat([
        grupo_especifico.drop_duplicates("product_id")["rating_num"].agg(["count", "mean"]),
        grupo_superior.drop_duplicates("product_id")["rating_num"].agg(["count", "mean"]),
        notas_comparacao.drop_duplicates("product_id")["rating_num"].agg(["count", "mean"]),
    ], axis=1, keys=[categoria_especifica.split("|")[-1], categoria_superior.split("|")[-1], "Conjunto geral"]).T
    comparacao_rating.columns = ["produtos_comparaveis", "media_observada"]
    comparacao_rating["produtos_comparaveis"] = comparacao_rating["produtos_comparaveis"].astype("Int64")
    display(comparacao_rating)
else:
    print("A comparação individual requer nova análise: quantidade de exceções, categoria ou outra nota do mesmo ID mudou.")

,1279
source_index,1279
product_id,B08L12N5H1
category,"Home&Kitchen|Kitchen&HomeAppliances|Vacuum,Cleaning&Ironing|Vacuums&FloorCare|Vacuums|HandheldVacuums"
rating,|
rating_num,


,product_id,rating_num


,produtos_comparaveis,media_observada
HandheldVacuums,7,3.942857
Vacuums,19,4.052632
Conjunto geral,1350,4.091852


In [43]:
observacoes["rating_analitico_num"] = observacoes["rating_num"].copy()
observacoes["rating_estimado"] = False
observacoes["metodo_estimativa_rating"] = pd.Series(pd.NA, index=observacoes.index, dtype="string")

if comparacao_rating is not None:
    grupo_elegivel = comparacao_rating.drop(index="Conjunto geral")
    grupo_elegivel = grupo_elegivel.loc[grupo_elegivel["produtos_comparaveis"].ge(minimo_comparaveis)].head(1)
    if not grupo_elegivel.empty:
        nota_estimada = round(grupo_elegivel["media_observada"].item(), 1)
        metodo = (f"Média de {grupo_elegivel['produtos_comparaveis'].item()} produtos em {grupo_elegivel.index.item()}; "
                  f"mínimo heurístico: {minimo_comparaveis}; arredondamento: uma casa decimal.")
        observacoes.loc[observacoes["rating_invalido"], "rating_analitico_num"] = nota_estimada
        observacoes.loc[observacoes["rating_invalido"], "rating_estimado"] = True
        observacoes.loc[observacoes["rating_invalido"], "metodo_estimativa_rating"] = metodo
    else:
        print("Nenhum dos dois níveis alcançou o mínimo de comparáveis; a nota permanece desconhecida.")


display(observacoes.loc[observacoes["rating_invalido"], ["product_id", "rating", "rating_num",
    "rating_analitico_num", "rating_estimado"]])

,product_id,rating,rating_num,rating_analitico_num,rating_estimado
1279,B08L12N5H1,|,<NA>,4.1,True


In [44]:
contagens_ausentes = observacoes.loc[observacoes["rating_count_num"].isna()].copy()
contagens_ausentes["nome_exibicao"] = contagens_ausentes["product_name"].str.slice(0, 65) + "…"
contagens_ausentes["reviews_disponiveis"] = contagens_ausentes["review_id"].str.split(",", regex=False).str.len()
contagens_ausentes["ocorrencias_id"] = contagens_ausentes["product_id"].map(observacoes["product_id"].value_counts())


display(contagens_ausentes[[
    "product_id", "rating_num", "rating_count", "reviews_disponiveis", "ocorrencias_id",
]])
referencias_contagem = observacoes.loc[
    observacoes["rating_count_num"].notna() & observacoes["category"].isin(contagens_ausentes["category"])
].sort_values("source_index").drop_duplicates(["category", "product_id"])

distribuicao_contagens = referencias_contagem.groupby("category")["rating_count_num"].describe()

display(distribuicao_contagens.set_axis(distribuicao_contagens.index.str.rsplit("|", n=1).str[-1]).round(2))

referencias_por_nota = referencias_contagem.groupby(["category", "rating_num"])["rating_count_num"].agg(["count", "median"])
comparacao_por_nota = contagens_ausentes[["category", "rating_num"]].drop_duplicates().merge(
    referencias_por_nota, on=["category", "rating_num"], how="left", validate="one_to_one"
)
comparacao_por_nota["count"] = comparacao_por_nota["count"].fillna(0).astype("Int64")

display(comparacao_por_nota[["rating_num", "count", "median"]])

,product_id,rating_num,rating_count,reviews_disponiveis,ocorrencias_id
282,B0B94JPY2N,3.0,<NA>,1,1
324,B0BQRJ3C47,5.0,<NA>,1,1


,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
USBCables,159.0,13972.77,25514.09,5.0,946.5,3049.0,13868.0,178817.0


,rating_num,count,median
0,3.0,0,<NA>
1,5.0,1,5.0


In [45]:
referencias_suficientes = distribuicao_contagens.loc[distribuicao_contagens["count"].ge(10)]
medianas_categoria = referencias_suficientes["50%"].round().astype("Int64")
metodos_contagem = (
    "Mediana de " + referencias_suficientes["count"].astype(int).astype(str)
    + " produtos na categoria " + referencias_suficientes.index.str.rsplit("|", n=1).str[-1]
    + "; primeira contagem válida por ID; arredondamento ao inteiro."
)

observacoes["rating_count_analitico_num"] = observacoes["rating_count_num"].copy()
observacoes["rating_count_estimado"] = (
    observacoes["rating_count"].isna() & observacoes["category"].isin(medianas_categoria.index)
)
observacoes.loc[observacoes["rating_count_estimado"], "rating_count_analitico_num"] = (
    observacoes.loc[observacoes["rating_count_estimado"], "category"].map(medianas_categoria)
)
observacoes["metodo_estimativa_rating_count"] = observacoes["category"].map(metodos_contagem).where(
    observacoes["rating_count_estimado"]
).astype("string")


display(observacoes.loc[observacoes["rating_count_estimado"], ["product_id", "rating_count",
    "rating_count_num", "rating_count_analitico_num", "rating_count_estimado"]])

,product_id,rating_count,rating_count_num,rating_count_analitico_num,rating_count_estimado
282,B0B94JPY2N,<NA>,<NA>,3049,True
324,B0BQRJ3C47,<NA>,<NA>,3049,True


In [46]:
pd.testing.assert_frame_equal(observacoes[fonte.columns], fonte)
assert observacoes["actual_price_num"].gt(0).all()
assert observacoes["discounted_price_num"].between(0, observacoes["actual_price_num"]).all()
assert observacoes["discount_percentage_num"].between(0, 100).all()
assert not observacoes[["desconto_inconsistente", "desconto_nao_avaliado"]].any().any()
for campo in ["rating", "rating_count"]:
    estimado = observacoes[campo + "_estimado"]
    assert observacoes.loc[estimado, campo + "_num"].isna().all()
    pd.testing.assert_series_equal(observacoes.loc[~estimado, campo + "_num"],
        observacoes.loc[~estimado, campo + "_analitico_num"], check_names=False)
    assert observacoes["metodo_estimativa_" + campo].notna().eq(estimado).all()
display(pd.DataFrame({"tipo": observacoes[[c + "_num" for c in campos_numericos_brutos]].dtypes.astype(str),
    "nulos_observados": observacoes[[c + "_num" for c in campos_numericos_brutos]].isna().sum()}))
display(observacoes.loc[observacoes["rating_estimado"] | observacoes["rating_count_estimado"],
    ["product_id", "metodo_estimativa_rating", "metodo_estimativa_rating_count"]].T.style.set_properties(**{"max-width": "300px", "white-space": "normal"}))

,tipo,nulos_observados
actual_price_num,Float64,0
discounted_price_num,Float64,0
discount_percentage_num,Float64,0
rating_num,Float64,1
rating_count_num,Int64,2


,282,324,1279
product_id,B0B94JPY2N,B0BQRJ3C47,B08L12N5H1
metodo_estimativa_rating,,,Média de 19 produtos em Vacuums; mínimo heurístico: 10; arredondamento: uma casa decimal.
metodo_estimativa_rating_count,Mediana de 159 produtos na categoria USBCables; primeira contagem válida por ID; arredondamento ao inteiro.,Mediana de 159 produtos na categoria USBCables; primeira contagem válida por ID; arredondamento ao inteiro.,


In [47]:
for c in ['rating_analitico_num','rating_estimado','metodo_estimativa_rating','rating_count_analitico_num','rating_count_estimado','metodo_estimativa_rating_count']:
    produtos[c] = produtos.source_index_referencia.map(observacoes.set_index('source_index')[c])
display(produtos.loc[produtos.rating_estimado | produtos.rating_count_estimado,
    ['product_id','rating_num','rating_analitico_num','rating_count_num','rating_count_analitico_num']])

,product_id,rating_num,rating_analitico_num,rating_count_num,rating_count_analitico_num
669,B08L12N5H1,<NA>,4.1,992,992
1257,B0B94JPY2N,3.0,3.0,<NA>,3049
1349,B0BQRJ3C47,5.0,5.0,<NA>,3049


### 8.5 — Exportação segura

O destino de teste é uma pasta exclusiva. Todos os CSVs são relidos com tipos explícitos e comparação exata antes da publicação da proposta.

In [48]:
exportar()

Contratos legados reconciliados (com cenários)


,antes,depois,depois_pct
reviews,9269.0,9269,100.00
titulo,207.0,207,2.23
conteudo,210.0,210,2.27
ambos,171.0,171,1.84
parcial,75.0,75,0.81
sem_texto,9023.0,9023,97.35
elegiveis,70.0,70,0.76
maior_confianca,70.0,70,0.76
exploratorio,101.0,101,1.09
conflitos,201.0,0,0.00


Proposta validada em C:\Users\Barbara Bertellini\Documents\Estudos\Pós Graduação\Fase 4\tech_challeng_amazon\backups\correcao_final_notebook_02_20261112_213042_nao_entrega\trabalho\proposta_02_9koqt93c


### 8.6 — Preparação para os Notebooks 03, 04 e 05

O Notebook 03 deve usar Produtos e Categorias. Os Notebooks 04 e 05 devem manter uma trilha por blocos da origem e distinguir os reviews com evidência individual. A projeção legada continuará atendendo os leitores atuais durante a migração.

### 8.7 — Preparação para o Power BI

Use Categoria → Produto com filtro unidirecional. Navegue de produto a review via Ocorrências e Vínculos. As redundâncias da ponte e as referências de proveniência não precisam virar relações ativas. Indicadores comerciais usam o grão de produto.

### 8.8 — Conclusões e limitações remanescentes

O modelo preserva as seis entidades e publica a recuperação com proveniência, status por campo e trilha de candidatos. A evidência cruzada amplia as hipóteses verificáveis e a estabilidade entre buscas reduz alguns erros de fronteira. A auditoria continua necessária para casos ambíguos, conflitantes ou com separadores ausentes. O estágio de maior confiança é um filtro estrutural. A revisão humana e a integração dos consumidores são etapas separadas.

As categorias de evidência podem se sobrepor. Os totais de divergência não medem erros de autoria. O comparativo da exportação usa como referência a proposta imediatamente anterior. O corpus agregado continua necessário para cobrir o catálogo.
